
![](https://raw.githubusercontent.com/wateraccounting/WaPORMOOC/main/images/banner_notebooks_WaPOR4Global.png)

[![](https://raw.githubusercontent.com//wateraccounting/WaPORMOOC/main/images/colab-badge.png)](https://colab.research.google.com/github/wateraccounting/WaPOR4Global/blob/main/4_Forecasting/Notebook_4_Field_Level_Forecasting.ipynb?target="_blank")

<div style="text-align:center; margin-top:15px;">
<h3 style="margin-bottom:5px;">
<i>Module 3.3 </i><b> Field-level Forecasting &mdash; Iraq</b>
</h3>

<p style="margin-top:5px;">
<b>Notebook 4:</b> Field-level forecasting  &nbsp; | &nbsp;
<b>Estimated time:</b> 2 hour &nbsp;
<b>Instructor:</b> Dr. Ahmed Elnaggar
</p>
</div>




---

### Course Information

| Item | Description |
|------|-------------|
| **Course** | MOOC — Global Challenges for Water in Agriculture |
| **Topic** | 3 of 3: Forecasting |
| **Focus** | Forecasting water consumption (AETI) and biomass (NPP) for 12 centre-pivot fields |
| **Region** | Erbil Governorate, Kurdistan Region of Iraq |
| **Data** | dekadal WaPOR observations (2018-2025) |
| **Models** | ARIMA, SARIMA, SARIMAX+PCP, SARIMAX+PCP&RET |

---

### Learning Objectives

By the end of this notebook you will be able to:

1. **Prepare** dekadal field-level time series for forecasting
2. **Apply** four forecasting models (ARIMA, SARIMA, SARIMAX+PCP, SARIMAX+PCP&RET) to predict water consumption
3. **Evaluate** forecast accuracy using RMSE, MAE and MAPE across 12 individual fields
4. **Compare** which exogenous variable (precipitation vs. reference ET) adds more predictive value
5. **Perform** short term forecasting beyond the observed data period
6. **Assess** long term forecast of water consumption for each field

---

### Table of Contents

1. The Final Zoom — From Governorate to Field
2. Setup & Data Loading
3. Field Exploration — 12 Centre Pivots
4. Preparing for Forecasting
5. Model 1 — ARIMA 
6. Model 2 — SARIMA (Seasonal Baseline)
7. Model 3 — SARIMAX + PCP (Precipitation)
8. Model 4 — SARIMAX + RET (Reference ET)
9. Model 5 — SARIMAX + PCP & RET
10. Cross-Field Model Comparison 
11. Future Forecasting — Beyond the Data
12. Long-term Climate Projection Benchmark -- SARIMA & MIROC6 (SSP5-8.5)
13. Spatial View — Mapping Forecast Quality & Future Outlook
14. Course Synthesis - The Three Zooms 

---

<a id="1"></a>
## 1. The Final Zoom — From Governorate to Field

> In **Topic 1** we surveyed Iraq's climate across 18 governorates — establishing the spatio-temporal characterization of the climate in Iraq.
>
> In **Topic 2** we zoomed into Erbil's cropland adding WaPOR satellite data to evaluate the relation between the climatic variables and agricultural water consumption and production.
>
> Now, in **Topic 3**, we reach the **operational level**: individual irrigated fields. We don't just evaluate the data — we use them to **look into the future**, predicting water consumption for each pivot from short to long term forecasts.

### Why Centre Pivots?

| Reason | Detail |
|--------|--------|
| **Defined boundary** | Circular footprint, easily delineated from satellite imagery |
| **Active management** | Irrigation decisions directly affect AETI |
| **Comparable units** | Similar size and shape for fair comparison |
| **Practical relevance** | Forecasting field-level water use supports irrigation scheduling |

### The WaPOR Variables

| Variable | WaPOR Name | Role in This Notebook |
|----------|------------|----------------------|
| **AETI** | Actual EvapoTranspiration & Interception | Target 1 — the water consumption we forecast |
| **NPP** | Net Primary Productivity | Target 2 - the biomass production we forecast |
| **PCP**| Precipitation| Exogeneous predictor |
| **RET**| Reference ET | Exogeneous predictor |

### The Four Models

| # | Model | Family | Key Feature |
|---|-------|--------|-------------|
| 1 | **ARIMA** | ARIMA | Basic autoregression forecasting |
| 2 | **SARIMA** | ARIMA | Handles trend + seasonality; our baseline |
| 3 | **SARIMAX + PCP** | ARIMA + exogenous | Does precipitation information improve the forecast? |
| 4 | **SARIMAX + PCP&RET** | ARIMA + exogenous | Does adding reference ET information improve the forecast? |

> The real questions: *Which fields are predictable? Does external information help? And what does the future hold?*

## 2. Setup ⚙️
## Import and install packages


In [ ]:
# =============================================================================
# INSTALL & IMPORT PACKAGES
# =============================================================================

# Uncomment to install if needed:
# !pip install numpy pandas matplotlib seaborn scipy statsmodels scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import os
import warnings
from datetime import datetime
from statsmodels.tools.sm_exceptions import ValueWarning
from statsmodels.graphics.tsaplots import plot_pacf, plot_acf


# --- Time Series & Forecasting ---
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error, r2_score

print(f"\n\u2705 Setup complete \u2014 Pandas {pd.__version__}, NumPy {np.__version__}")
print(f"\U0001f4c5 Analysis date: {datetime.now().strftime('%Y-%m-%d')}")

---
### Data Files

Create folders for where to upload the input data and where to store the results.



In [2]:
# =============================================================================
# CREATE INPUT AND OUTPUT FOLDERS
# =============================================================================
data_folder = 'data'
os.makedirs(data_folder, exist_ok=True)

output_folder = 'output_data'
os.makedirs(output_folder, exist_ok=True)

Place your data in the `data/` folder:

| File | Variable | Scope | Description |
|------|----------|-------|-------------|
| `Erbil_L1-PCP-D_mm_per_dekad.csv` | **PCP** | **Erbil-wide** | Dekadal precipitation (mm) — single climate series |
| `Erbil_L1-RET-D_mm_per_dekad.csv` | **RET** | **Erbil-wide** | Dekadal reference ET (mm) — single climate series |
| `Pivots_L3-AETI-D_mm_per_dekad` | AETI | per-pivot | Dekadal actual ET (mm) — 12 pivots |
| `Pivots_L3-NPP-D_gc_per_m2_per_dekad`  | NPP  | per-pivot | Dekadal net primary production (gC/m²) — 12 pivots |

**Climate CSV format:** Two columns — date (`YYYY-MM-DD`) and the dekadal value (the source files have an unnamed header row, which the loader handles automatically). Because precipitation and reference ET are forced by the regional atmosphere, we treat them as a **single Erbil-wide series broadcast to every pivot** — they are climate drivers, not field-specific signals.

**Pivot CSV format:** First column = date (`DD/MM/YYYY`), remaining columns = pivot IDs (14, 21, 30, 33, 35, 40, 51, 78, 84, 99, 119, 151, 163).



In [ ]:
# Upload data from Notebook_1
from google.colab import files
uploaded = files.upload()

# Save each file to the destination folder
for filename, content in uploaded.items():
    dest_path = f'/content/data/{filename}'
    with open(dest_path, 'wb') as f:
        f.write(content)
    print(f" Saved: {dest_path}")

---

<a id="3"></a>
### 3. Field Exploration — 12 Centre Pivots

> *Before forecasting, let's understand each field's water consumption, biomass production, and how the two variables relate to each other.*

First we define a function to read the pivot data, followed by reading the two files.

In [5]:
def load_pivot_data(filepath, var_name):
    """Load dekadal pivot-level CSV (one column per pivot)."""
    df = pd.read_csv(filepath)
    date_col = df.columns[0]

    for fmt in ['%d/%m/%Y', '%m/%d/%Y', '%Y-%m-%d', '%Y/%m/%d']:
        try:
            df[date_col] = pd.to_datetime(df[date_col], format=fmt)
            break
        except (ValueError, TypeError):
            continue
    else:
        df[date_col] = pd.to_datetime(df[date_col], dayfirst=True)

    df.set_index(date_col, inplace=True)
    df.index.name = 'date'
    df = df.sort_index()
    df.columns = [str(c).strip() for c in df.columns]
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    print(f"📄 {var_name}: {df.shape[0]} dekads × {df.shape[1]} pivots | "
          f"{df.index.min().strftime('%b %Y')} — {df.index.max().strftime('%b %Y')}")
    return df

In [ ]:
# =============================================================================
# READ DATA — DEKADAL PIVOT-LEVEL DATA
# =============================================================================

print("="*70)
print("LOADING DEKADAL DATA FOR CENTRE PIVOTS")
print("="*70)

df_aeti = load_pivot_data('/content/data/Pivots_L3-AETI-D_mm_per_dekad.csv', 'AETI')
df_npp  = load_pivot_data('/content/data/Pivots_L3-NPP-D_gc_per_m2_per_dekad.csv', 'NPP')

pivots = df_aeti.columns.tolist()
n_pivots = len(pivots)
print(f"\n📍 {n_pivots} centre pivots: {pivots}")




In [ ]:
# =============================================================================
# INSPECT DATA — DEKADAL PIVOT-LEVEL DATA
# =============================================================================

print("="*70)
print("AETI \u2014 FIRST 10 DEKADS")
print("="*70)
display(df_aeti.head(10))

print("\n\U0001f4ca Summary Statistics - AETI (mm/dekad) by Pivot:")
aeti_stats = df_aeti.describe().T
aeti_stats['CV (%)'] = (aeti_stats['std'] / aeti_stats['mean'] * 100).round(1)
display(aeti_stats[['mean', 'std', 'CV (%)', 'min', 'max']].round(2))

# Missing values
for name, df in [('AETI', df_aeti), ('NPP', df_npp)]:
    m = df.isnull().sum().sum()
    sym = '\u2705' if m == 0 else '\u26a0\ufe0f'
    print(f"{sym} {name}: {m} missing values")

🔎 Inspect the NPP data as well, which center pivot has the highest mean NPP value?

Lets plot the timeseries for each of the center pivots.

In [ ]:
# =============================================================================
# AETI TIME SERIES - ALL PIVOTS (Small Multiples)
# =============================================================================

n_cols = 3
n_rows = (n_pivots + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3 * n_rows), sharex=True)
fig.suptitle('Water Consumption (AETI) \u2014 Centre-Pivot Fields',
             fontsize=16, fontweight='bold', y=1.01)

axes_flat = axes.flatten()
colors = plt.cm.tab20(np.linspace(0, 1, n_pivots))

for idx, (pivot, color) in enumerate(zip(pivots, colors)):
    ax = axes_flat[idx]
    ax.fill_between(df_aeti.index, 0, df_aeti[pivot], color=color, alpha=0.3)
    ax.plot(df_aeti[pivot], color=color, linewidth=0.8)
    ax.set_title(f'Pivot {pivot}', fontsize=10, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.2)

for idx in range(n_pivots, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.text(0.5, -0.01, 'Date', ha='center', fontsize=11)
fig.text(-0.01, 0.5, 'AETI (mm/dekad)', va='center', rotation='vertical', fontsize=11)
plt.tight_layout()
plt.show()

📈 How does water consumption (AETI) align with biomass production (NPP)?

Let's plot the dual time series for a sample of four center pivots to find out.

In [ ]:
# =============================================================================
# TWO-VARIABLE OVERVIEW - AETI + NPP (selected pivots)
# =============================================================================

# Show 4 pivots (first, last, and two middle)
showcase_idx = [0, 5, 9, 12]
showcase = [pivots[i] for i in showcase_idx]

fig, axes = plt.subplots(len(showcase), 1, figsize=(16, 3.5 * len(showcase)), sharex=True)
fig.suptitle('AETI,  NPP \u2014 Selected Pivots', fontsize=16, fontweight='bold', y=1.01)

for i, pivot in enumerate(showcase):
    ax = axes[i]
    ax2 = ax.twinx()

    l1, = ax.plot(df_aeti[pivot], linewidth=1, label='AETI', color='blue')
    l2, = ax2.plot(df_npp[pivot], linewidth=0.8, alpha=0.7, label='NPP', color='green')

    ax.set_ylabel('AETI (mm)', color='blue')
    ax2.set_ylabel('NPP (gC/m\u00b2)', color='green')
    ax.set_title(f'Pivot {pivot}', fontweight='bold', loc='left')
    ax.legend(handles=[l1, l2], loc='upper right', fontsize=8, ncol=3)
    ax.grid(True, alpha=0.2)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

💧 How does water use vary across all fields?

Let's use a box plot to compare the AETI distribution across every center pivot.

In [ ]:
# =============================================================================
# CROSS-FIELD COMPARISON - BOX PLOT
# =============================================================================

fig, axes = plt.subplots(1, 1, figsize=(16, 6))
fig.suptitle('Comparing Water Use Across Pivots', fontsize=14, fontweight='bold')

# Box plot
ax = axes
medians = df_aeti.median()
pivots = medians.index.tolist()
bp = ax.boxplot([df_aeti[p].dropna() for p in pivots],
                  tick_labels=[f'P{p}' for p in pivots],
                  patch_artist=True, showfliers=True,
                  flierprops={'marker': '.', 'markersize': 2, 'alpha': 0.3})
cmap_box = ['skyblue'] * n_pivots # Set all boxes to a single color
for patch, color in zip(bp['boxes'], cmap_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('AETI (mm/dekad)')
ax.set_title('AETI Distribution', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CORRELATION:  AETI vs NPP - PER PIVOT
# =============================================================================

print("="*70)
print("HOW STRONGLY DOES NPP RELATE TO AETI?")
print("="*70)

corr_data = []
for pivot in pivots:
    r_npp  = df_aeti[pivot].corr(df_npp[pivot])
    corr_data.append({'Pivot': pivot, 'AETI~NPP': r_npp})

corr_df = pd.DataFrame(corr_data).set_index('Pivot')

# Visualise
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(n_pivots)
width = 0.35
ax.bar(x + width/2, corr_df['AETI~NPP'], width, color='#2ca02c', alpha=0.7, label='AETI ~ NPP')
ax.set_xticks(x)
ax.set_xticklabels([f'P{p}' for p in pivots], fontsize=9)
ax.set_ylabel('Pearson Correlation (r)')
ax.legend(fontsize=10)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=corr_df['AETI~NPP'].mean(), color='red', linestyle='--', linewidth=1, label='Average Correlation') # Line of tendency
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

print(f"\n\U0001f4a1 Average |r|: NPP = {corr_df['AETI~NPP'].abs().mean():.3f}")

### Correlation: AETI vs Erbil-Wide Climate Drivers (PCP & RET)

Let's examine how AETI correlates with the regional climate drivers, Precipitation (PCP) and Reference Evapotranspiration (RET).

---

### 3a. Erbil Climate Exploration — PCP & RET


> Before zooming into the 12 pivots, here is the **regional climate data** the fields are operating in. Precipitation (PCP) and reference ET (RET) come from the WaPOR L1 dekadal product over the Erbil governorate and represent the **atmospheric forcing** common to every pivot.

First we define a function to read the climate data, followed by reading the two files.

In [12]:
# =============================================================================
# DATA LOADING — ERBIL-WIDE CLIMATE DRIVERS
# =============================================================================

def load_climate_series(filepath, var_name):
    raw = pd.read_csv(filepath, header=0)          # row `,0` becomes the header
    raw.columns = ['date', var_name]
    raw['date'] = pd.to_datetime(raw['date'], errors='coerce')
    raw = raw.dropna(subset=['date']).set_index('date').sort_index()
    s = raw[var_name]

    print(f"🌦️  {var_name} (Erbil-wide): {len(s)} dekads | "
          f"{s.index.min().strftime('%b %Y')} — {s.index.max().strftime('%b %Y')} | "
          f"mean = {s.mean():.2f}")
    return s

In [ ]:
# =============================================================================
# READ DATA — ERBIL-WIDE CLIMATE DRIVERS
# =============================================================================

print("="*70)
print("LOADING ERBIL-WIDE CLIMATE DRIVERS (PCP & RET)")
print("="*70)

pcp_erbil = load_climate_series('/content/data/Erbil_L1-PCP-D_mm_per_dekad.csv', 'PCP')
ret_erbil = load_climate_series('/content/data/Erbil_L1-RET-D_mm_per_dekad.csv', 'RET')

# Align climate series to the AETI dekadal index (drop / forward-fill any gaps)
pcp_erbil = pcp_erbil.reindex(df_aeti.index)
ret_erbil = ret_erbil.reindex(df_aeti.index)
df_climate = pd.DataFrame({'PCP': pcp_erbil, 'RET': ret_erbil})

print(f"🌧️  Climate frame: {df_climate.shape[0]} dekads × {df_climate.shape[1]} variables (PCP, RET)")

Lets plot the full time series and the mean dekadal values in two separate graphs.

In [ ]:
# =============================================================================
# ERBIL CLIMATE CONTEXT — PCP (bars) + RET (line), with dekadal climatology
# =============================================================================

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=False)

# --- Panel 1: full dekadal time series ---
ax1 = axes[0]
ax1.bar(pcp_erbil.index, pcp_erbil.values, width=8,
        color='#2c7fb8', alpha=0.7, label='PCP (mm/dekad)')
ax1.set_ylabel('Precipitation (mm/dekad)', color='#2c7fb8')
ax1.tick_params(axis='y', labelcolor='#2c7fb8')
ax1.set_ylim(bottom=0)

ax1b = ax1.twinx()
ax1b.plot(ret_erbil.index, ret_erbil.values,
          color='#d95f0e', linewidth=1.4, label='RET (mm/dekad)')
ax1b.set_ylabel('Reference ET (mm/dekad)', color='#d95f0e')
ax1b.tick_params(axis='y', labelcolor='#d95f0e')
ax1b.set_ylim(bottom=0)

ax1.set_title('Erbil dekadal climate — full record', fontweight='bold')
ax1.grid(True, alpha=0.2)

# --- Panel 2: dekadal climatology (mean by dekad-of-year) ---
doy_idx_pcp = np.minimum((pcp_erbil.index.dayofyear.values - 1) // 10, 35)
doy_idx_ret = np.minimum((ret_erbil.index.dayofyear.values - 1) // 10, 35)
doy_pcp = pcp_erbil.groupby(doy_idx_pcp).mean()
doy_ret = ret_erbil.groupby(doy_idx_ret).mean()

ax2 = axes[1]
x = np.arange(len(doy_pcp))
ax2.bar(x, doy_pcp.values, color='#2c7fb8', alpha=0.7, label='Mean PCP')
ax2.set_ylabel('Mean PCP (mm/dekad)', color='#2c7fb8')
ax2.tick_params(axis='y', labelcolor='#2c7fb8')
ax2.set_xlabel('Dekad of year (1–36)')
ax2b = ax2.twinx()
ax2b.plot(x, doy_ret.values, color='#d95f0e', linewidth=2, marker='o', markersize=4,
          label='Mean RET')
ax2b.set_ylabel('Mean RET (mm/dekad)', color='#d95f0e')
ax2b.tick_params(axis='y', labelcolor='#d95f0e')
ax2.set_title('Dekadal climatology (mean over 2018–2025)', fontweight='bold')
ax2.set_xticks(x[::3])
ax2.set_xticklabels([f'{d+1}' for d in x[::3]])
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print(f"💧 Annual PCP (mean):   {pcp_erbil.resample('YE').sum().mean():.0f} mm/yr")
print(f"☀️  Annual RET (mean):   {ret_erbil.resample('YE').sum().mean():.0f} mm/yr")
print(f"📉 Aridity (PCP / RET): {pcp_erbil.sum() / ret_erbil.sum():.2f}  "
      f"(< 0.2 → arid, 0.2–0.5 → semi-arid)")


In [ ]:
# =============================================================================
# CORRELATION: AETI vs PCP and AETI vs RET - PER PIVOT
# =============================================================================

print("="*70)
print("HOW STRONGLY DOES AETI RELATE TO REGIONAL PCP & RET?")
print("="*70)

corr_data_climate = []
for pivot in pivots:
    # Assuming df_climate (PCP, RET) is already aligned to df_aeti's index
    r_pcp = df_aeti[pivot].corr(df_climate['PCP'])
    r_ret = df_aeti[pivot].corr(df_climate['RET'])
    corr_data_climate.append({'Pivot': pivot, 'AETI~PCP': r_pcp, 'AETI~RET': r_ret})

corr_df_climate = pd.DataFrame(corr_data_climate).set_index('Pivot')

# Visualise
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(n_pivots)
width = 0.35

ax.bar(x - width/2, corr_df_climate['AETI~PCP'], width, color='#2c7fb8', alpha=0.7, label='AETI ~ PCP')
ax.bar(x + width/2, corr_df_climate['AETI~RET'], width, color='#d95f0e', alpha=0.7, label='AETI ~ RET')

ax.set_xticks(x)
ax.set_xticklabels([f'P{p}' for p in pivots], fontsize=9)
ax.set_ylabel('Pearson Correlation (r)')
ax.set_title('AETI Correlation with Regional Climate Drivers (PCP & RET)', fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

print(f"\n\U0001f4a1 Average |r|: PCP = {corr_df_climate['AETI~PCP'].mean():.3f}")
print(f"\U0001f4a1 Average |r|: RET = {corr_df_climate['AETI~RET'].mean():.3f}")

This analysis shows a bigger influence of RET on AETI compared to PCP. And PCP having a negative relation with AETI, compared to a positive relationship between RET and AETI. Repeat the analyses to see how the climate factors influence NPP.


In [ ]:
# =============================================================================
# CORRELATION: NPP vs PCP and NPP vs RET - PER PIVOT
# =============================================================================





 ## 💡 Quiz:

* Which pivot has the highest correlation (r) between NPP and **PCP**? What is r in that case (two digits)?

* Which pivot has the highest correlation (r) between NPP and **RET**? What is r in that case (two digits)?

✍ Make sure to write them down, as you will need them for the upcoming quiz

---

<a id="4"></a>
## 4. Preparing for Forecasting



### Train-Test Split Strategy

> A common approach for training and evaluating a model is to split the dataset into two, whereby the first part is used to train the model and the second part to evaluate the model. A typical distribution between the training and testing data is 70-30. We hold out the **last two years of data (72 dekads)** as the test set (25%).



In [ ]:
# =============================================================================
# SPLIT DATA SET INTO TRAIN / TEST
# =============================================================================

FORECAST_HORIZON = 72   # 2 years of dekads

split_idx = len(df_aeti) - FORECAST_HORIZON

train_aeti = df_aeti.iloc[:split_idx]
test_aeti  = df_aeti.iloc[split_idx:]

train_npp = df_npp.iloc[:split_idx]
test_npp  = df_npp.iloc[split_idx:]

# --- Erbil-wide climate drivers (shared across all pivots) ---
train_climate = df_climate.iloc[:split_idx]
test_climate  = df_climate.iloc[split_idx:]

print(f"📅 Train: {train_aeti.index.min().strftime('%b %Y')} — "
      f"{train_aeti.index.max().strftime('%b %Y')} ({len(train_aeti)} dekads)")
print(f"📅 Test:  {test_aeti.index.min().strftime('%b %Y')} — "
      f"{test_aeti.index.max().strftime('%b %Y')} ({len(test_aeti)} dekads)")
print(f"🌧️ Climate exogenous frame ready: train {train_climate.shape}, test {test_climate.shape}")

print("\n✅ Split complete.")

### Evaluation Metrics
Several metrics are then calculated to evaluate the model using the function defined in the next cell:

| Stat | Description |Unit|
|------|----------|-------------|
| r^2 | Coefficient of determination | -|
| RMSE | Root mean square error|mm/month|
| Bias | Bias | mm/month |
| PBias | percentage bias | % |

In [ ]:
# =============================================================================
# METRIC FUNCTIONS
# =============================================================================

def compute_metrics(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)
    mask = ~(np.isnan(actual) | np.isnan(predicted))
    actual, predicted = actual[mask], predicted[mask]

    # 1. Calculate R-squared (R2) using sklearn's r2_score
    r2 = r2_score(actual, predicted)

    # 2. Calculate RMSE using scikit-learn
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    # 3. Calculate Bias
    bias = np.mean(predicted - actual)

    # 4. Calculate Percentage Bias (PBIAS)
    sum_diff = np.sum(predicted - actual)
    sum_actual = np.sum(actual)

    if sum_actual != 0:
        pbias = (sum_diff / sum_actual) * 100
    else:
        pbias = np.nan # Avoid division by zero

    return {'r2': round(r2, 3), 'RMSE': round(rmse, 3), 'Bias': round(bias, 1), 'PBias':round(pbias, 1)}

print("\n\u2705 Metric functions defined.")

Now we are ready to apply the forecasting models.

---

<a id="5b"></a>
## 5. Model 1 — ARIMA (Non-Seasonal Baseline)

> **ARIMA** (AutoRegressive Integrated Moving Average) is a forecasting model that uses a series' own past values and past forecast errors to project its future values, without needing external variables. It is described by three parameters $(p, d, q)$, which combine autoregression, differencing, and moving average components.

An <a href="https://machinelearningplus.com/time-series/arima-model-time-series-forecasting-python/" target="_blank">ARIMA model</a> is fully defined by these three parameters:

| Parameter | Name | Meaning |
|---|---|---|
| $p$ | Order of the AR term | number of autoregressive lags |
| $d$ | Order of differencing | number of times the series must be differenced to become stationary |
| $q$ | Order of the MA term | number of lagged forecast errors included in the model |

In the following sections we will determine the right value for each of these parameters, by following the same general workflow used for any ARIMA model:

| # | Step | Description |
|---|------|-------------|
| 1 | Identify | Plot the time series and understand its behavior |
| 2 | Check Stationarity | Use ADF test and plots. If non-stationary, difference the data |
| 3 | Identify (p, d, q) | Examine ACF and PACF plots to suggest values for p and q |
| 4 | Estimate | Fit the ARIMA model using the identified parameters |
| 5 | Diagnose | Check residuals for white noise (no autocorrelation) |
| 6 | Forecast | Use the validated model to forecast future values |

Learn more about ARIMA model ➤ <a href="https://otexts.com/fpp2/arima.html" target="_blank">here</a>

<p align="center">
  <img src="https://raw.githubusercontent.com/wateraccounting/WaPORMOOC/main/images/arima.PNG" height="450" style="vertical-align:top;">
</p>

## Stationarity test
The value of d, is the minimum number of differencing needed to make the series stationary. And if the time series is already stationary, then **d = 0.**

A well used approach to determine if a timeseries is stationary is the <a href="https://machinelearningplus.com/time-series/augmented-dickey-fuller-test/" target="_blank">Augmented Dickey-Fuller test</a> (ADF)

<p align="CENTER">
  <img src="https://raw.githubusercontent.com/wateraccounting/WaPORMOOC/main/images/arima4.JPG" height="450" style="vertical-align:top";>
</p>


In [ ]:
# =============================================================================
# STATIONARITY CHECK FOR AETI - ADF TEST
# =============================================================================

print("="*70)
print("AUGMENTED DICKEY-FULLER TEST - AETI BY PIVOT")
print("="*70)
print(f"{'Pivot':<10} {'ADF Stat':<12} {'p-value':<12} {'Stationary?':<12}")
print("-"*46)

for pivot in pivots:
    series = df_aeti[pivot].dropna()
    result = adfuller(series, autolag='AIC')
    sym = '\u2705 Yes' if result[1] < 0.05 else '\u274c No'
    print(f"P{pivot:<9} {result[0]:<12.2f} {result[1]:<12.3f} {sym}")

Based on the stationarity test it shows the AETI data is suitable for forecasting using ARIMA and the parameter `d` should be set at 0. Can you check if the NPP data is also stationary?

In [ ]:
# =============================================================================
# STATIONARITY CHECK FOR NPP - ADF TEST
# =============================================================================

print("="*70)
print("AUGMENTED DICKEY-FULLER TEST - NPP BY PIVOT")
print("="*70)
print(f"{'Pivot':<10} {'ADF Stat':<12} {'p-value':<12} {'Stationary?':<12}")
print("-"*46)

for pivot in pivots:
    series = df_npp[pivot].dropna()
    result = adfuller(series, autolag='AIC')
    sym = '\u2705 Yes' if result[1] < 0.05 else '\u274c No'
    print(f"P{pivot:<9} {result[0]:<12.2f} {result[1]:<12.3f} {sym}")

For the other two parameters we evaluate the time series for partial auto-correlation (PACF) and auto-correlation (ACF). We can do this by using `statsmodels` libary and plot the first differential timeseries followed by evaluating the PACF and ACF.

<p align="CENTER">
  <img src="https://raw.githubusercontent.com/wateraccounting/WaPORMOOC/main/images/arima5.JPG" height="350" style="vertical-align:top";>
</p>

In [ ]:
# =============================================================================
# EVALUATING THE PARTIAL AUTOCORRELATION AND AUTOCORRELATION
# =============================================================================


pivot_id = pivots[1]
pivot_data = df_aeti[pivot_id]

plt.rcParams.update({'figure.figsize':(9,3), 'figure.dpi':120})

fig, axes = plt.subplots(1, 1, sharex=True)
axes.plot(pivot_data.diff()); axes.set_title(f'1st Differencing - Pivot {pivot_id}')

plot_pacf(pivot_data.diff().dropna())
plot_acf(pivot_data.diff().dropna())

plt.show()

From the results it shows that the PACF and ACF lag 1 and 2 are significant (above the shaded blue area). We therefore set the values for p and q at `2`.


| Parameter | Value | Meaning |
|---|---|---|
| $p$ | 2 | number of autoregressive lags |
| $d$ | 0 | number of times the series must be differenced to become stationary |
| $q$ |2| number of lagged forecast errors included in the model |

> We use a simple configuration: `(2,0,2)`. This model serves as a baseline to understand the performance without seasonal considerations, allowing a direct comparison to the SARIMA model where seasonality is explicitly handled (Model 2).

The following two cells run the ARIMA model, calculate the metrics and present the results of the observed data versus the validation data in graphs.




In [ ]:
# =============================================================================
# MODEL 1: ARIMA - FIT & VALIDATE AETI FOR ALL PIVOTS (Non-Seasonal)
# =============================================================================


print("="*70)
print("FITTING ARIMA(2,0,2) TO ALL PIVOTS")
print("="*70)

arima_results = {}
order_arima = (2, 0, 2)

for i, pivot in enumerate(pivots):
    print(f"   Fitting pivot {pivot} ({i+1}/{n_pivots})...", end='', flush=True)
    train = train_aeti[pivot].dropna()
    test  = test_aeti[pivot].dropna()

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=ValueWarning)
            warnings.simplefilter("ignore", category=FutureWarning)

            # ARIMA model without seasonal_order
            model = SARIMAX(train, order=order_arima,
                            enforce_stationarity=False, enforce_invertibility=False)
            fit = model.fit(disp=False, maxiter=300)
            validation = fit.forecast(steps=len(test))#.clip(lower=0)

        validation.index = test.index
        metrics = compute_metrics(test.values, validation.values)
        arima_results[pivot] = {'validation': validation, 'metrics': metrics,
                                 'aic': round(fit.aic, 1)}

    except Exception as e:
        print(f"\n   ☠️ Pivot {pivot}: {e}")
        arima_results[pivot] = {'validation': None, 'metrics': None, 'aic': None}

print("\n\n✅ ARIMA complete!")
print(f"\n{'Pivot':<10} {'r2':<10} {'RMSE':<10} {'Bias':<10} {'PBias':<10}")
print("-"*40)

#print(arima_results)

for p in pivots:
  r = arima_results[p]
  if r['metrics']:
        m = r['metrics']
        print(f"P{p:<9} {m['r2']:<10} {m['RMSE']:<10} {m['Bias']:<10} {m['PBias']:<10}")

In [ ]:
# =============================================================================
# ARIMA FORECAST PLOTS
# =============================================================================

n_cols_p = 3
n_rows_p = (n_pivots + n_cols_p - 1) // n_cols_p

fig, axes = plt.subplots(n_rows_p, n_cols_p, figsize=(18, 3.5 * n_rows_p))
fig.suptitle('ARIMA Forecasts - All Pivots', fontsize=16, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

for idx, pivot in enumerate(pivots):
    ax = axes_flat[idx]
    r = arima_results[pivot]
    ctx = 0 #max(0, split_idx - 72)
    ax.plot(df_aeti.index[ctx:split_idx], df_aeti[pivot].iloc[ctx:split_idx],
            color='steelblue', linewidth=0.8, label='Train')
    ax.plot(test_aeti.index, test_aeti[pivot], color='black', linewidth=1.5, label='Actual')

    if r['validation'] is not None:
        ax.plot(r['validation'].index, r['validation'].values,
                color='green', linewidth=1.5, linestyle='--', label='ARIMA')
        ax.set_title(f"P{pivot} (r2={r['metrics']['r2']:.2f})", fontsize=10, fontweight='bold')
    else:
        ax.set_title(f'P{pivot} (failed)', fontsize=10, fontweight='bold')

    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.2)

axes_flat[0].legend(fontsize=7, loc='upper left')
for idx in range(n_pivots, len(axes_flat)):
    axes_flat[idx].set_visible(False)
plt.tight_layout()
plt.show()

> 📉 Based on the results of the **ARIMA** model it appears that ARIMA is not the most suitable forecasting approach for the type of time series we are dealing with. The weak performance comes down to **seasonality** in the data, which a non-seasonal model simply can't represent.
>
> ⚠️ Setting `p` or `d` to 36 (one full year, since this is dekadal data) does improve the metrics on paper, but for the wrong reasons: it's overfitting to the cycle length rather than genuinely modeling seasonality.

🔜 **Next step:** move to the **SARIMA** model, which adds the seasonal terms ARIMA is missing.

---

<a id="5"></a>
## 6. Model 2 — SARIMA (Seasonal Baseline)

To incorporate the seasonal aspects of the time series we apply the SARIMA model instead.

> **SARIMA** $(p,d,q)(P,D,Q)_s$ combines autoregression, differencing and moving average — including seasonal counterparts.
>
<p align="CENTER">
  <img src="https://raw.githubusercontent.com/wateraccounting/WaPORMOOC/main/images/arima_vs_sarima.png" height="450" style="vertical-align:top";>
</p>

>
> We use a simple, reliable configuration: `(1,0,1)(1,1,0,36)`. Where 36 relates to the lenght of a season. This is the **baseline** to forecast AETI that all other models compete against.

The following two cells run the model and present the results.

In [ ]:
# =============================================================================
# MODEL 2: SARIMA - FIT & VALIDATE AETI FOR ALL PIVOTS
# =============================================================================

import warnings
from statsmodels.tools.sm_exceptions import ValueWarning

print("="*70)
print("FITTING SARIMA(1,0,1)(1,0,1,36) TO ALL PIVOTS")
print("="*70)

SEASONAL_PERIOD = 36   # 36 dekads per year
print(f"🔄 Seasonal period: {SEASONAL_PERIOD} dekads (1 year)")

sarima_results = {}
order = (1, 0, 1)
seasonal_order = (1, 0, 1, SEASONAL_PERIOD)
FORECAST_HORIZON = 36  # 1 year of dekads
forecast_dates = pd.date_range(start=df_aeti.index[-1] + pd.Timedelta(days=10),
                               periods=FORECAST_HORIZON, freq='10D')

for i, pivot in enumerate(pivots):
    print(f"\r   Fitting pivot {pivot} ({i+1}/{n_pivots})...", end='', flush=True)
    train = train_aeti[pivot].dropna()
    test  = test_aeti[pivot].dropna()

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=ValueWarning)
            warnings.simplefilter("ignore", category=FutureWarning)

            model = SARIMAX(train, order=order, seasonal_order=seasonal_order,
                            enforce_stationarity=False, enforce_invertibility=False)
            fit = model.fit(disp=False, maxiter=300)

            # --- Validation: forecast over the test period (used for metrics) ---
            validation = fit.forecast(steps=len(test)).clip(lower=0)
            validation.index = test.index

            metrics = compute_metrics(test.values, validation.values)
            sarima_results[pivot] = {'validation': validation,
                                     'metrics': metrics,
                                     'aic': round(fit.aic, 1)}

            # =================================================================
            # FORECAST  (uncomment the 4 lines below to forecast beyond data)
            # =================================================================
            #fc = fit.get_forecast(steps=FORECAST_HORIZON)
            #forecast_mean = fc.predicted_mean.clip(lower=0)
            #forecast_mean.index = forecast_dates
            #sarima_results[pivot]['forecast'] = {'mean': forecast_mean}

            # -----------------------------------------------------------------
            # 95% CONFIDENCE INTERVAL  (uncomment together with the forecast)
            # -----------------------------------------------------------------
            #forecast_ci = fc.conf_int()
            #forecast_ci.index = forecast_dates
            #sarima_results[pivot]['forecast']['ci'] = forecast_ci

    except Exception as e:
        print(f"\n   ⚠️ Pivot {pivot}: {e}")
        sarima_results[pivot] = {'validation': None, 'metrics': None, 'aic': None}

print("\n\n✅ SARIMA complete!")
print(f"\n{'Pivot':<10} {'r2':<10} {'RMSE':<10} {'Bias':<10} {'PBias':<10}")
print("-"*50)
for p in pivots:
    r = sarima_results[p]
    if r['metrics']:
        m = r['metrics']
        print(f"P{p:<9} {m['r2']:<10} {m['RMSE']:<10} {m['Bias']:<10} {m['PBias']:<10}")

In [ ]:
# =============================================================================
# SARIMA RESULTS PLOTS
# =============================================================================

n_cols_p = 3
n_rows_p = (n_pivots + n_cols_p - 1) // n_cols_p

fig, axes = plt.subplots(n_rows_p, n_cols_p, figsize=(18, 3.5 * n_rows_p))
fig.suptitle('SARIMA Results - All Pivots', fontsize=16, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

for idx, pivot in enumerate(pivots):
    ax = axes_flat[idx]
    r = sarima_results[pivot]

    ctx = 0  # max(0, split_idx - 72)
    ax.plot(df_aeti.index[ctx:split_idx], df_aeti[pivot].iloc[ctx:split_idx],
            color='steelblue', linewidth=0.8, label='Train')
    ax.plot(test_aeti.index, test_aeti[pivot],
            color='black', linewidth=1.5, label='Actual')

    if r.get('validation') is not None:
        ax.plot(r['validation'].index, r['validation'].values,
                color='#d62728', linewidth=1.5, linestyle='--', label='SARIMA')
        ax.set_title(f"P{pivot} (r2={r['metrics']['r2']:.2f})", fontsize=10, fontweight='bold')
    else:
        ax.set_title(f'P{pivot} (failed)', fontsize=10, fontweight='bold')

    # =========================================================================
    # FORECAST  (uncomment after enabling the forecast in the fit cell above)
    # =========================================================================
    #fc = r.get('forecast')
    #if fc is not None:
        #ax.plot(forecast_dates, fc['mean'].values,
                #color='#2ca02c', linewidth=1.5, linestyle='--', label='Forecast')

    # -------------------------------------------------------------------------
    # 95% CONFIDENCE INTERVAL  (uncomment together with the forecast above)
    # -------------------------------------------------------------------------
    #fc = r.get('forecast')
    #if fc is not None and 'ci' in fc:
        #ci = fc['ci']
        #ax.fill_between(forecast_dates, ci.iloc[:, 0].values, ci.iloc[:, 1].values,
                        #color='#2ca02c', alpha=0.15, label='95% CI')

    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.2)

axes_flat[0].legend(fontsize=7, loc='upper left')
for idx in range(n_pivots, len(axes_flat)):
    axes_flat[idx].set_visible(False)
plt.tight_layout()
plt.show()

> ✅ **Result:** The SARIMA model successfully captures the seasonality present in the historical observations.

To forecast beyond the available data:

1. Uncomment the **`forecast`** section in both cells above.
2. Re-run both cells.

> 🤔 **Think about it:** Why dont we calculate the performance metrics for this period?

Once you're ready to see the uncertainty around each forecast, uncomment the **confidence interval (`ci`)** section as well, which will plot each pivot's forecast together with its 95% confidence interval.

---

<a id="7"></a>
## 7. Model 3 — SARIMAX + PCP (Precipitation)

We also know that AETI (or NPP) is dependent on climatic factors (e.g. PCP and RET). This will be further explored in the models presented in this section, starting with precipitation.

> **SARIMAX** extends SARIMA with exogenous (X) variables. Here we add **PCP** (precipitation) as a predictor: precipitation influences the water availability which directly constrains how much water crops can transpire.
>
> During the validation period, we use observed PCP (simulating a scenario where precipitation is measured or already forecast).

In [ ]:
# =============================================================================
# MODEL 3: SARIMAX + PCP
# =============================================================================

print("="*70)
print("FITTING SARIMAX(1,0,1)(1,0,1,36) + PCP TO ALL PIVOTS")
print("="*70)

sarimax_pcp_results = {}

for i, pivot in enumerate(pivots):
    print(f"\r   Fitting pivot {pivot} ({i+1}/{n_pivots})...", end='', flush=True)
    train_y = train_aeti[pivot].dropna()
    test_y  = test_aeti[pivot].dropna()
    train_x = train_climate['PCP'].reindex(train_y.index).ffill().bfill().to_frame() # Ensure DataFrame
    test_x  = test_climate['PCP'].reindex(test_y.index).ffill().bfill().to_frame() # Ensure DataFrame

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=ValueWarning)
            warnings.simplefilter("ignore", category=FutureWarning)

            model = SARIMAX(train_y, exog=train_x, order=order,
                            seasonal_order=seasonal_order,
                            enforce_stationarity=False, enforce_invertibility=False)
            fit = model.fit(disp=False, maxiter=300)
            validation = fit.forecast(steps=len(test_y), exog=test_x.values).clip(lower=0)
            validation.index = test_y.index
            metrics = compute_metrics(test_y.values, validation.values)
            sarimax_pcp_results[pivot] = {'validation': validation, 'metrics': metrics,
                                            'aic': round(fit.aic, 1)}
    except Exception as e:
        print(f"\n   \u26a0\ufe0f Pivot {pivot}: {e}")
        sarimax_pcp_results[pivot] = {'validation': None, 'metrics': None, 'aic': None}

print("\n\n\u2705 SARIMAX + PCP complete!")
print(f"\n{'Pivot':<10} {'r2':<10} {'RMSE':<10} {'Bias':<10} {'PBias':<10}")
print("-"*40)
for p in pivots:
    r = sarimax_pcp_results[p]
    if r['metrics']:
        m = r['metrics']
        print(f"P{p:<9} {m['r2']:<10} {m['RMSE']:<10} {m['Bias']:<10} {m['PBias']:<10}")

In [ ]:
# =============================================================================
# SARIMAX + PCP FORECAST PLOTS
# =============================================================================

n_cols_p = 3
n_rows_p = (n_pivots + n_cols_p - 1) // n_cols_p

fig, axes = plt.subplots(n_rows_p, n_cols_p, figsize=(18, 3.5 * n_rows_p))
fig.suptitle('SARIMAX + PCP Forecasts - All Pivots', fontsize=16, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

for idx, pivot in enumerate(pivots):
    ax = axes_flat[idx]
    r = sarimax_pcp_results[pivot]
    ctx = 0 #max(0, split_idx - 72)
    ax.plot(df_aeti.index[ctx:split_idx], df_aeti[pivot].iloc[ctx:split_idx],
            color='steelblue', linewidth=0.8, label='Train')
    ax.plot(test_aeti.index, test_aeti[pivot], color='black', linewidth=1.5, label='Actual')
    if r['validation'] is not None:
        ax.plot(r['validation'].index, r['validation'].values,
                color='#9467bd', linewidth=1.5, linestyle='--', label='SARIMAX+PCP')
        ax.set_title(f"P{pivot} (r2={r['metrics']['r2']:.2f})", fontsize=10, fontweight='bold')
    else:
        ax.set_title(f'P{pivot} (failed)', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.2)

axes_flat[0].legend(fontsize=7, loc='upper left')
for idx in range(n_pivots, len(axes_flat)):
    axes_flat[idx].set_visible(False)
plt.tight_layout()
plt.show()

You noticed that for this model we are only showing the validation and not the forecasted data. What could be the reason for this?

<details>
<summary>💡 Click to reveal the answer</summary>

Since the **SARIMAX** model depends on precipitation as an input variable, forecasting beyond the observed period would require *future* PCP values and only the historical PCP record is available.

> We'll come back to this limitation later in the notebook.

</details>

---

<a id="8"></a>
## 8. Model 4 — SARIMAX + RET (Reference ET)

> Now we swap PCP for **RET** (Reference ET) as the exogenous variable. The logic: RET determines the water requirement for the crop.
>
> 🎯 **Key question:** *Can reference ET (RET) predict water consumption (AETI) better than precipitation (PCP)?*

In [ ]:
# =============================================================================
# MODEL 4: SARIMAX + RET
# =============================================================================


In [ ]:
# =============================================================================
# SARIMAX + RET FORECAST PLOTS
# =============================================================================



📈 Which model is better at forecasting the AETI data for pivot 52?

 a) SARIMA

 b) SARIMAX + PCP

 c) SARIMAX + RET

Provide the r2, RSME, Bias and PBias for SARIMAX + RET for pivot 52. Note down your answer, you need it for the QUIZ! 💡

What could be one of the reasons pivot 164 is not able to succesfully forecast AETI in the dry summer season?

<details>
<summary> Click to reveal the answer</summary>

>>  This pivot appears not to be cropped in the two seasons in the test period.
</details>

---

<a id="8b"></a>
## 9. Model 5 — SARIMAX + PCP + RET

Because the wet season is more dominated by the rainfall availability and the dry season by the atmospheric water demand, the last model we use both the PCP and RET as drivers.
>
> Both PCP and RET come from the WaPOR L1 dekadal product over the **Erbil governorate**. They are the kind of variables a national meteorological service or a global model (e.g. downscaled CMIP6 projections like MIROC6) actually delivers, so a model that performs well with them is one that can be operationalised in real water-management workflows.
>
We fit a single 2-column exogenous matrix `[PCP, RET]` — identical across all 12 pivots — and let SARIMAX learn each pivot's individual sensitivity.


In [ ]:
# =============================================================================
# MODEL 5: SARIMAX + PCP + RET (Erbil-wide climate exogenous, shared)
# =============================================================================

print("="*70)
print("FITTING SARIMAX(1,0,1)(1,0,1,36) + [PCP, RET] TO ALL PIVOTS")
print("="*70)

sarimax_climate_results = {}

for i, pivot in enumerate(pivots):
    print(f"\r   Fitting pivot {pivot} ({i+1}/{n_pivots})...", end='', flush=True)
    train_y = train_aeti[pivot].dropna()
    test_y  = test_aeti[pivot].dropna()
    train_x = train_climate.reindex(train_y.index).ffill().bfill()
    test_x  = test_climate.reindex(test_y.index).ffill().bfill()

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=ValueWarning)
            warnings.simplefilter("ignore", category=FutureWarning)
            model = SARIMAX(train_y, exog=train_x, order=order,
                            seasonal_order=seasonal_order,
                            enforce_stationarity=False, enforce_invertibility=False)
            fit = model.fit(disp=False, maxiter=300)
            validation = fit.forecast(steps=len(test_y), exog=test_x.values).clip(lower=0)
            validation.index = test_y.index
            metrics = compute_metrics(test_y.values, validation.values)
            sarimax_climate_results[pivot] = {'validation': validation, 'metrics': metrics,
                                              'aic': round(fit.aic, 1), 'fit': fit}
    except Exception as e:
        print(f"\n   ⚠️ Pivot {pivot}: {e}")
        sarimax_climate_results[pivot] = {'validation': None, 'metrics': None, 'aic': None, 'fit': None}

print("\n\n✅ SARIMAX + PCP + RET complete!")
print(f"\n{'Pivot':<10} {'r2':<10} {'RMSE':<10} {'Bias':<10} {'PBias':<10}")
print("-"*40)
for p in pivots:
    r = sarimax_climate_results[p]
    if r['metrics']:
        m = r['metrics']
        print(f"P{p:<9} {m['r2']:<10} {m['RMSE']:<10} {m['Bias']:<10} {m['PBias']:<10}")

In [ ]:
# =============================================================================
# SARIMAX + PCP + RET FORECAST PLOTS
# =============================================================================

n_cols_p = 3
n_rows_p = (n_pivots + n_cols_p - 1) // n_cols_p

fig, axes = plt.subplots(n_rows_p, n_cols_p, figsize=(18, 3.5 * n_rows_p))
fig.suptitle('SARIMAX + PCP + RET Results - All Pivots', fontsize=16, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

for idx, pivot in enumerate(pivots):
    ax = axes_flat[idx]
    r = sarimax_climate_results[pivot]
    ctx = 0 #max(0, split_idx)
    ax.plot(df_aeti.index[ctx:split_idx], df_aeti[pivot].iloc[ctx:split_idx],
            color='steelblue', linewidth=0.8, label='Train')
    ax.plot(test_aeti.index, test_aeti[pivot], color='black', linewidth=1.5, label='Actual')
    if r['validation'] is not None:
        ax.plot(r['validation'].index, r['validation'].values,
                color='#17becf', linewidth=1.5, linestyle='--', label='SARIMAX+PCP+RET')
        ax.set_title(f"P{pivot} (r2={r['metrics']['r2']:.2f})", fontsize=10, fontweight='bold')
    else:
        ax.set_title(f'P{pivot} (failed)', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.2)

axes_flat[0].legend(fontsize=7, loc='upper left')
for idx in range(n_pivots, len(axes_flat)):
    axes_flat[idx].set_visible(False)
plt.tight_layout()
plt.show()

---

<a id="9"></a>
## 10. Cross-Field Model Comparison

Lets compare the results of all 4 models and for all pivots to answer the following three key questions:
1. **Which model performs best overall?** (across all fields)
2. **Which fields are easiest / hardest to forecast?** (consistent across models)
3. **PCP vs. RET** — which exogenous variable adds more predictive value?

📝 NOTE: uncomment the SARIMAX+RET lines to include the results of the SARIMAX+RET model

In [ ]:
# =============================================================================
# CONSOLIDATED COMPARISON TABLE — All MODELS and PIVOTS
# =============================================================================

all_results = {
    'SARIMA':        sarima_results,
    'SARIMAX+PCP':  sarimax_pcp_results,
    #'SARIMAX+RET':   sarimax_ret_results,
    'SARIMAX+PCP+RET': sarimax_climate_results,
}
model_names = list(all_results.keys())
model_colors = {
    'SARIMA':          '#d62728',
    'SARIMAX+PCP':    '#9467bd',
    #'SARIMAX+RET':     '#2ca02c',
    'SARIMAX+PCP+RET': '#17becf',
}

rows = []
for pivot in pivots:
    row = {'Pivot': f'P{pivot}',
           'Mean AETI': round(df_aeti[pivot].mean(), 2),
           'CV (%)': round(df_aeti[pivot].std() / df_aeti[pivot].mean() * 100, 1)}
    for mn, res in all_results.items():
        r = res[pivot]
        if r['metrics']:
            row[f'{mn} r2'] = r['metrics']['r2']
            row[f'{mn} RMSE'] = r['metrics']['RMSE']
            row[f'{mn} Bias']  = r['metrics']['Bias']
            row[f'{mn} PBias'] = r['metrics']['PBias']
        else:
            row[f'{mn} r2'] = np.nan
            row[f'{mn} RMSE'] = np.nan
            row[f'{mn} Bias']  = np.nan
            row[f'{mn} PBias'] = np.nan
    rows.append(row)

comp_df = pd.DataFrame(rows).set_index('Pivot')
r2_cols = [f'{mn} r2' for mn in model_names]
comp_df['Best Model'] = comp_df[r2_cols].idxmin(axis=1).str.replace(' r2', '')
comp_df['Avg r2'] = comp_df[r2_cols].mean(axis=1).round(2)

print("="*100)
print("CROSS-FIELD MODEL COMPARISON — r2")
print("="*100)
display(comp_df[[c for c in comp_df.columns if 'r2' in c or c in ['Mean AETI', 'CV (%)', 'Best Model']]])

comp_df.to_csv(os.path.join(data_folder, 'pivot_forecast_comparison.csv'))
print("\n✅ Saved to data/pivot_forecast_comparison.csv")


In [ ]:
# =============================================================================
# MODEL COMPARISON VISUALISATION — 2 PANELS
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
fig.suptitle('Cross-Field validation Comparison',
             fontsize=16, fontweight='bold')

pivot_labels = [f'P{p}' for p in pivots]
x = np.arange(n_pivots)
n_models = len(model_names)
width = 0.85 / n_models   # squeeze 5 bars per group

# --- Panel 1: r2 grouped bars ---
ax1 = axes[0]
for i, mn in enumerate(model_names):
    vals = comp_df[f'{mn} r2'].values
    ax1.bar(x + i * width, vals, width, color=model_colors[mn], alpha=0.8, label=mn)
ax1.set_xticks(x + (n_models - 1) * width / 2)
ax1.set_xticklabels(pivot_labels, rotation=45, fontsize=8)
ax1.set_ylabel('r2')
ax1.set_title('r2 by Pivot & Model (higher = better)', fontweight='bold')
ax1.legend(fontsize=8, ncol=2)
ax1.grid(True, alpha=0.2, axis='y')

# --- Panel 2: r2 heatmap ---
ax2 = axes[1]
r2_table = comp_df[r2_cols].copy()
r2_table.columns = model_names
sns.heatmap(r2_table, annot=True, fmt='.2f', cmap='RdYlGn',
            ax=ax2, linewidths=0.5, cbar_kws={'label': 'r2', 'shrink': 0.7})
ax2.set_title('r2 Heatmap — All Models × All Pivots', fontweight='bold')
ax2.set_ylabel('')
ax2.tick_params(axis='x', rotation=25, labelsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# DETAILED VALIDATION VIEW - SELECTED PIVOTS (all models)
# =============================================================================

avg_r2 = comp_df['Avg r2'].sort_values(ascending=False)
best_p   = avg_r2.index[0]
median_p = avg_r2.index[len(avg_r2) // 2]
worst_p  = avg_r2.index[-1]

#showcase = [(avg_r2.index[0], "pivot1"), (avg_r2.index[4], "pivot2"), (avg_r2.index[5], "pivot3")]
showcase = [(best_p, 'Best (highest r2)'),
           (median_p, 'Median'),
            (worst_p, 'Worst (lowest r2)')]

fig, axes = plt.subplots(3, 1, figsize=(16, 15), sharex=True)
fig.suptitle('Detailed Validation Comparison - All Models',
             fontsize=16, fontweight='bold', y=1.01)

for idx, (plabel, tlabel) in enumerate(showcase):
    ax = axes[idx]
    pid = plabel.replace('P', '')
    ctx = max(0, split_idx - 72)

    ax.plot(df_aeti.index[ctx:split_idx], df_aeti[pid].iloc[ctx:split_idx],
            color='steelblue', linewidth=0.8, alpha=0.5, label='Training')
    ax.plot(test_aeti.index, test_aeti[pid], color='black', linewidth=2, label='Actual')

    for mn, res, color, ls in [
        ('SARIMA', sarima_results, '#d62728', '--'),
        ('SARIMAX+PCP', sarimax_pcp_results, '#ff7f0e', '-.'),
        #('SARIMAX+RET', sarimax_ret_results, '#9467bd', ':'),
        ('SARIMAX+PCP+RET', sarimax_climate_results, '#2ca02c', (0, (3, 1, 1, 1)))
    ]:
        r = res[pid]
        if r['validation'] is not None:
            r2_score = r['metrics']['r2']
            ax.plot(r['validation'].index, r['validation'].values,
                    color=color, linewidth=1.5, linestyle=ls,
                    label=f'{mn} ({r2_score:.2f})')

    ax.set_ylabel('AETI (mm/dekad)')
    ax.set_title(f'{tlabel}: {plabel} (Avg r2 = {avg_r2[plabel]:.2f})',
                 fontweight='bold', loc='left', fontsize=11)
    ax.legend(fontsize=8, loc='upper right', ncol=3)
    ax.grid(True, alpha=0.2)
    ax.axvline(x=df_aeti.index[split_idx], color='gray', linewidth=1, linestyle='-', alpha=0.4)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

🔎 The worst performing pivots are unable to predict the low AETI values in some years for the dry season. This is most likely related to the fact that the farmer is not growing any crops that year, which can not be predicted (the center pivots which were considered for the analyses were selected based on consistency in cropped area). Using the models for forecasting therefore does not provide information related to what will actually happen.

---

<a id="10a"></a>
## 10a. Short-Term Operational Forecast — Rolling Forecast on the Full Record

> Sections 5–9 evaluated the models against a held-out test set (`test_aeti`, the last two years) to check accuracy. For actual operational use we don't want to hold anything back — we retrain on the **entire observed record** and forecast forward from today.
>
> SARIMA is used here because, unlike the SARIMAX variants, it needs no exogenous inputs and can therefore forecast beyond the data immediately (Section 11 shows how the exogenous-driven models get around this by forecasting PCP/RET first).

Change `ROLLING_PIVOT` and `ROLLING_HORIZON` below to get an up-to-date forecast for any pivot and any number of dekads ahead — re-running the cell always refits on the complete dataset, so the forecast stays current as new data comes in.

In [ ]:
# =============================================================================
# SHORT-TERM ROLLING FORECAST — SARIMA TRAINED ON FULL OBSERVED RECORD
# =============================================================================

ROLLING_PIVOT   = '36'     # change to any pivot ID, e.g. '52'
ROLLING_HORIZON = 12            # dekads ahead to forecast (12 dekads ≈ 4 months)

def rolling_forecast(pivot, horizon, order=order, seasonal_order=seasonal_order):
    """
    Refit SARIMA on the FULL observed AETI series for one pivot (no train/test
    split) and forecast `horizon` dekads beyond the last observation.
    Because it retrains on 100% of the data every time it is called, this can
    be re-run at any point to produce an up-to-date operational forecast for
    any period.
    """
    series = df_aeti[pivot].dropna()

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ValueWarning)
        warnings.simplefilter("ignore", category=FutureWarning)

        model = SARIMAX(series, order=order, seasonal_order=seasonal_order,
                        enforce_stationarity=False, enforce_invertibility=False)
        fit = model.fit(disp=False, maxiter=300)

        forecast_dates_roll = pd.date_range(start=series.index[-1] + pd.Timedelta(days=10),
                                            periods=horizon, freq='10D')
        forecast = fit.get_forecast(steps=horizon)
        mean = forecast.predicted_mean.clip(lower=0); mean.index = forecast_dates_roll
        ci   = forecast.conf_int();                   ci.index   = forecast_dates_roll

    return {'fit': fit, 'mean': mean, 'ci': ci}

print("="*70)
print(f"ROLLING FORECAST — PIVOT {ROLLING_PIVOT}, {ROLLING_HORIZON} DEKADS AHEAD")
print("="*70)

rolling_result = rolling_forecast(ROLLING_PIVOT, ROLLING_HORIZON)
print(f"   ✅ Trained on full record: {df_aeti.index.min().strftime('%b %Y')} — "
      f"{df_aeti.index.max().strftime('%b %Y')} ({len(df_aeti)} dekads)")
print(f"   📅 Forecast: {rolling_result['mean'].index[0].date()} → "
      f"{rolling_result['mean'].index[-1].date()}")

# =============================================================================
# ROLLING FORECAST PLOT
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 5))
ctx = max(0, len(df_aeti) - 108)
ax.plot(df_aeti.index[ctx:], df_aeti[ROLLING_PIVOT].iloc[ctx:],
        color='steelblue', linewidth=1, label='Observed (full record)')

mean, ci = rolling_result['mean'], rolling_result['ci']
ax.plot(mean.index, mean.values, color='#d62728', linewidth=1.5,
        linestyle='--', label='Rolling forecast')
lower_col, upper_col = ci.columns[0], ci.columns[1]
ax.fill_between(ci.index, ci[lower_col].clip(lower=0), ci[upper_col],
                color='#d62728', alpha=0.15, label='95% CI')

ax.axvline(x=df_aeti.index[-1], color='gray', linewidth=0.8, alpha=0.5)
ax.set_title(f'P{ROLLING_PIVOT} — Rolling Forecast ({ROLLING_HORIZON} dekads ahead)',
             fontweight='bold')
ax.set_ylabel('AETI (mm/dekad)')
ax.set_xlabel('Date')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=30)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

print("\n💡 Re-run this cell with a new ROLLING_PIVOT / ROLLING_HORIZON at any time —")
print("   it always retrains on the full observed record, so the forecast stays current.")

---

<a id="10"></a>
## 11. Future Forecasting — Beyond the Data

So far only for the SARIMA model we were able to forecast beyond the available data.

To be able to forecast using the SARIMA models which incorporate exogenous variables (e.g. SARIMA+PCP, SARIMA+RET etc), these variables require forecasting first. For the purpose of the exercise we will forecast PCP and RET using a univariate SARIMA model on the full record.

Alternatively, forecasted data from climate forecasting sites (e.g. European Centre for Medium-Range Weather Forecasts (ECMWF)) can be used as well.


In [ ]:
# -----------------------------------------------------------------------------
# Forecast Erbil-wide PCP & RET (univariate SARIMA on full record)
# -----------------------------------------------------------------------------

FORECAST_HORIZON = 36  # 1 year of dekads

last_date = df_aeti.index[-1]
forecast_dates = pd.date_range(start=last_date + pd.Timedelta(days=10),
                             periods=FORECAST_HORIZON, freq='10D')

print("\n" + "="*70)
print("FORECASTING ERBIL-WIDE PCP & RET 1 YEAR AHEAD")
print("="*70)

forecast_climate = {}
for var_name in ['PCP', 'RET']:
    series = df_climate[var_name].dropna()

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ValueWarning)
        warnings.simplefilter("ignore", category=FutureWarning)

        model = SARIMAX(series, order=order, seasonal_order=seasonal_order,
                    enforce_stationarity=False, enforce_invertibility=False)
        fit = model.fit(disp=False, maxiter=300)
        forecast = fit.get_forecast(steps=FORECAST_HORIZON)
        mean = forecast.predicted_mean.clip(lower=0); mean.index = forecast_dates
        ci   = forecast.conf_int();                   ci.index   = forecast_dates

    forecast_climate[var_name] = {'mean': mean, 'ci': ci}
    print(f"   ✅ {var_name}: forecast horizon {forecast_dates[0].date()} → {forecast_dates[-1].date()}, "
          f"projected annual total = {mean.sum():.0f} mm")

forecast_exog = pd.DataFrame({'PCP': forecast_climate['PCP']['mean'],
                            'RET': forecast_climate['RET']['mean']})

# =============================================================================
#  PLOT PCP & RET FORECASTS (Erbil)
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle('Erbil Climate Drivers — 1-Year-Ahead Forecast (95% CI)',
             fontsize=14, fontweight='bold')

for ax, var_name, hist, color in [
    (axes[0], 'PCP', df_climate['PCP'], '#2c7fb8'),
    (axes[1], 'RET', df_climate['RET'], '#d95f0e'),
]:
    ctx = max(0, len(hist) - 72)
    ax.plot(hist.index[ctx:], hist.values[ctx:],
            color=color, linewidth=1, label=f'Observed {var_name}')
    fc = forecast_climate[var_name]
    ax.plot(fc['mean'].index, fc['mean'].values,
            color='black', linewidth=1.5, linestyle='--', label='Forecast')
    lower, upper = fc['ci'].columns[0], fc['ci'].columns[1]
    ax.fill_between(fc['ci'].index,
                    fc['ci'][lower].clip(lower=0), fc['ci'][upper],
                    color=color, alpha=0.18, label='95% CI')
    ax.axvline(x=hist.index[-1], color='gray', linewidth=0.8, alpha=0.5)
    ax.set_title(f'{var_name} — projected annual total: {fc["mean"].sum():.0f} mm',
                 fontweight='bold')
    ax.set_ylabel(f'{var_name} (mm/dekad)')
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

---

### Future Climate Drivers & Climate-Driven AETI

> First we plot the **1-year-ahead forecasts of Erbil PCP and RET themselves** (univariate SARIMA on the full record). These projected climate fields are then fed as exogenous inputs to SARIMAX to produce a **climate-driven future AETI** for every pivot — the orange curve in the second figure. Comparing it to the original univariate SARIMA forecast (red) shows how much the climate signal shifts our expectation of future field-scale water consumption.


In [ ]:
# =============================================================================
# FORECAST AETI SARIMAX+PCP+RET
# =============================================================================
forecast_climate_results = {}

for i, pivot in enumerate(pivots):
    print(f"\r   Fitting pivot {pivot} ({i+1}/{n_pivots})...", end='', flush=True)
    train_y = train_aeti[pivot].dropna()
    train_x = train_climate.reindex(train_y.index).ffill().bfill()

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=ValueWarning)
            warnings.simplefilter("ignore", category=FutureWarning)

            model = SARIMAX(train_y, exog=train_x, order=order,
                            seasonal_order=seasonal_order,
                            enforce_stationarity=False, enforce_invertibility=False)
            fit = model.fit(disp=False, maxiter=300)
            forecast = fit.get_forecast(steps=FORECAST_HORIZON, exog=forecast_exog.values)
            mean = forecast.predicted_mean.clip(lower=0); mean.index = forecast_dates
            ci   = forecast.conf_int();                   ci.index   = forecast_dates
        forecast_climate_results[pivot] = {'mean': mean, 'ci': ci}

    except Exception as e:
        print(f"\n   ⚠️ Pivot {pivot}: {e}")
        forecast_climate_results[pivot] = {'mean': None, 'ci': None}

In [ ]:
# =============================================================================
# AETI FORECAST SARIMAX+PCP+RET PLOTS (with confidence intervals)
# =============================================================================

fig, axes = plt.subplots(n_rows_p, n_cols_p, figsize=(18, 3.5 * n_rows_p))
fig.suptitle('AETI Forecast - 1 Year Ahead (with 95% CI)',
             fontsize=16, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

for idx, pivot in enumerate(pivots):
    ax = axes_flat[idx]
    ctx = max(0, len(df_aeti) - 72)
    ax.plot(df_aeti.index[ctx:], df_aeti[pivot].iloc[ctx:],
            color='steelblue', linewidth=1, label='Observed')

    fc = forecast_climate_results[pivot]
    if fc['mean'] is not None:
        ax.plot(fc['mean'].index, fc['mean'].values,
                color='#d62728', linewidth=1.5, linestyle='--', label='Forecast')
        ci = fc['ci']
        lower_col = ci.columns[0]
        upper_col = ci.columns[1]
        ax.fill_between(ci.index, ci[lower_col].clip(lower=0), ci[upper_col],
                        color='#d62728', alpha=0.15, label='95% CI')
        total_future = fc['mean'].sum()
    else:
        total_future = 0

    ax.axvline(x=df_aeti.index[-1], color='gray', linestyle='-', linewidth=0.8, alpha=0.5)
    ax.set_title(f"P{pivot} (proj. {total_future:.0f} mm/yr)", fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.2)
    ax.set_ylim(bottom=0)

axes_flat[0].legend(fontsize=7, loc='upper left')
for idx in range(n_pivots, len(axes_flat)):
    axes_flat[idx].set_visible(False)
plt.tight_layout()
plt.show()

print("💡 The shaded bands show 95% confidence intervals \u2014")
print("   wider bands = less certainty about the forecast.")

---

<a id="11b"></a>
## 11.5 Long-term Climate Projection Benchmark -- SARIMA vs MIROC6 (SSP5-8.5)

> Short-horizon SARIMA forecasts are great for the next few months -- but drought planning, irrigation infrastructure design, and crop-mix decisions need an outlook that runs **decades** ahead. For that horizon, statistical models alone are not enough: we need a **physics-based climate projection** that knows about CO2 forcing, land-atmosphere feedbacks, and regional downscaling.

Both files are available in the `data/` folder of this notebook's GitHub repository: the future projection `data/Erbil_PCP_and_RET_projection.csv` (2015-2099, SSP5-8.5) is generated from the downscaled MIROC6 model, while the historical run `Erbil_PCP_and_RET_projection_historical.csv` (1950-2014) is historical climate data, used here as the reference period for bias correction.

*MIROC6 is one of three CMIP6 models validated for Iraq's climate by [Mukheef et al. (2024)](https://www.sciencedirect.com/science/article/pii/S0169809524002527), which found strong performance (R² 0.95–0.99) after bias correction for stations in central/west Iraq.*

**⬇️ Upload both files in the cell below**

| Source | Description |
|---|---|
| **MIROC6** | A coupled atmosphere-ocean general-circulation model contributed to CMIP6 by JAMSTEC / U-Tokyo / NIES (Japan). Recommended for Iraq because of its skill in reproducing the eastern Mediterranean / Mesopotamian precipitation regime. |
| **Downscaling** | Bias-corrected statistical downscaling to ~25 km, then aggregated dekadally, matching the WaPOR L1 grid. |
| **SSP5-8.5** | The high-emission "fossil-fuelled development" Shared Socioeconomic Pathway -- the upper end of plausible 21st-century forcing, used here as a conservative (worst-case) drought-planning scenario. |
| **PET via Penman-Monteith** | Computed from the downscaled MIROC6 daily fields (`tasmin`, `tasmax`, `sfcWind`, `rsds`, `hurs`) using the FAO-56 PM equation, then summed to dekads. PET is the reference (short-grass) ET;|

### What you can expect in this section:

- Upload the future projection and historical MIROC6 files
- Read daily csv files and convert to dekadal values (future projection + historical run)
- Plot the full MIROC6 trajectory 2026-2099 with 10-year rolling means.
- Compare historical PCP and RET with MIROC6 data, by dekad-of-year
- Bias correct MIROC6 data
- Run SARIMAX+PCP+RET for AETI projection, using the bias-corrected PCP/RET
- Analyse results

In [ ]:
# -----------------------------------------------------------------------------
# Upload the MIROC6 climate projection data used in this section
# -----------------------------------------------------------------------------
from google.colab import files
uploaded = files.upload()

# Save each file to the destination folder
for filename, content in uploaded.items():
    dest_path = f'/content/data/{filename}'
    with open(dest_path, 'wb') as f:
        f.write(content)
    print(f" Saved: {dest_path}")

In [ ]:
# =============================================================================
# READ CLIMATE PROJECTED DATA - Aggregate to dekadal data (future projection + historical run)
# =============================================================================

def _dekad_start(d):
    day = 1 if d.day <= 10 else (11 if d.day <= 20 else 21)
    return pd.Timestamp(d.year, d.month, day)

def load_miroc6_dekadal(csv_path):
    raw = pd.read_csv(csv_path)
    raw['time'] = pd.to_datetime(raw['time'], errors='coerce')
    raw = raw.dropna(subset=['time']).sort_values('time')
    raw['date'] = raw['time'].apply(_dekad_start)
    dekadal = raw.groupby('date').agg(PCP_proj=('Precipitation', 'sum'),
                                      RET_proj=('Penman_Monteith', 'sum')).round(3)
    print(f'   Aggregated {len(raw):,} daily rows -> {len(dekadal):,} dekads.')
    return dekadal

# --- Future projection (2015-2099, SSP5-8.5) ---
PROJ_CSV = os.path.join('/content/data/Erbil_PCP_and_RET_projection.csv')
proj = load_miroc6_dekadal(PROJ_CSV)
print(f'MIROC6 SSP5-8.5 projection loaded: {len(proj):,} dekads, '
      f'{proj.index.min().date()} -> {proj.index.max().date()}')
display(proj.head(3).round(2))

# --- Historical run (1950-2014, needed for bias correction below) ---
HIST_CSV = os.path.join('/content/data/Erbil_PCP_and_RET_projection_historical.csv')
proj_hist = load_miroc6_dekadal(HIST_CSV)
print(f'MIROC6 historical run loaded: {len(proj_hist):,} dekads, '
      f'{proj_hist.index.min().date()} -> {proj_hist.index.max().date()}')
display(proj_hist.head(3).round(2))

In [ ]:
#----
# Plot Full MIROC6 projection
# ----

annual = proj.resample('YE').sum()  # annual totals
annual.columns = ['PCP (mm/yr)', 'RET (mm/yr)']
annual['Climatic water deficit (RET - PCP)'] = annual['RET (mm/yr)'] - annual['PCP (mm/yr)']
roll = annual.rolling(10, min_periods=5).mean()

fig, ax = plt.subplots(figsize=(14, 5.5))
ax.plot(annual.index, annual['PCP (mm/yr)'], color='#1f77b4', alpha=0.4, lw=0.8, label='PCP annual')
ax.plot(roll.index,   roll['PCP (mm/yr)'],   color='#1f77b4', lw=2.2, label='PCP 10-yr rolling mean')
ax.plot(annual.index, annual['RET (mm/yr)'], color='#ff7f0e', alpha=0.4, lw=0.8, label='RET annual')
ax.plot(roll.index,   roll['RET (mm/yr)'],   color='#ff7f0e', lw=2.2, label='RET 10-yr rolling mean')
ax.set_title(f'Shamamuk -- MIROC6 SSP5-8.5 annual PCP & RET, '
             f'{proj.index.min().year}-{proj.index.max().year}', fontweight='bold')
ax.set_ylabel('mm / year'); ax.set_xlabel('Year'); ax.grid(True, alpha=0.25)
ax.legend(loc='upper left', fontsize=9, ncol=2); plt.tight_layout(); plt.show()

### Compare historical climate data vs MIROC6 future proyection --By dekad of year

📌 NOTE: `proj_hist` (historical climate data, 1950-2014) and `proj` (the MIROC6
 future projection, 2015-2099) do not share calendar years, so we compare
 the *climatological* seasonal cycle of each (mean by dekad-of-year)

In [ ]:
# =============================================================================
# COMPARE HISTORICAL CLIMATE DATA vs MIROC6 FUTURE PROJECTION -- BY DEKAD-OF-YEAR
# =============================================================================

def dekad_of_year(index):
    return (index.month - 1) * 3 + np.where(index.day <= 10, 1, np.where(index.day <= 20, 2, 3))


print(f"Historical climate data (proj_hist): {proj_hist.index.min().date()} -> {proj_hist.index.max().date()}")
print(f"MIROC6 future projection (proj): {proj.index.min().date()} -> {proj.index.max().date()}")

hist_all   = proj_hist.copy()
future_all = proj.copy()

hist_all['dekad']   = dekad_of_year(hist_all.index)
future_all['dekad'] = dekad_of_year(future_all.index)

hist_by_dekad   = hist_all.groupby('dekad')[['PCP_proj', 'RET_proj']].mean()
future_by_dekad = future_all.groupby('dekad')[['PCP_proj', 'RET_proj']].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_by_dekad.index, hist_by_dekad['PCP_proj'], color='#1f77b4',
            linestyle='--', label='Historical (1950-2014)')
axes[0].plot(future_by_dekad.index, future_by_dekad['PCP_proj'], color='#595AE3',
            label='MIROC6 projection (2015-2099)')
axes[0].set_title('PCP -- mean by dekad-of-year', fontweight='bold')
axes[0].set_xlabel('Dekad of year (1-36)'); axes[0].set_ylabel('mm/dekad')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.25)

axes[1].plot(hist_by_dekad.index, hist_by_dekad['RET_proj'], color='#96E38D',
            linestyle='--', label='Historical (1950-2014)')
axes[1].plot(future_by_dekad.index, future_by_dekad['RET_proj'], color='#5B9453',
            label='MIROC6 projection (2015-2099)')
axes[1].set_title('RET -- mean by dekad-of-year', fontweight='bold')
axes[1].set_xlabel('Dekad of year (1-36)'); axes[1].set_ylabel('mm/dekad')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

### Compare observed WaPOR (2018-2025) vs MIROC6 for the same period

🗓️ `proj` (MIROC6's projection, 2015-2099) overlaps with the real WaPOR record
(2018-2025) -- this overlap is what lets us do a genuine bias correction:
 compare what MIROC6 simulated for 2018-2025 against what WaPOR actually
 observed in that same window, by dekad-of-year.

In [ ]:
# =============================================================================
# COMPARE OBSERVED WaPOR (2018-2025) vs MIROC6 FOR THE SAME PERIOD -- REAL BIAS
# =============================================================================

overlap_start = max(proj.index.min(), df_climate.index.min())
overlap_end   = min(proj.index.max(), df_climate.index.max())
print(f"Overlap window (MIROC6 projection vs WaPOR observed): "
      f"{overlap_start.date()} -> {overlap_end.date()}")

proj_overlap = proj.loc[overlap_start:overlap_end].copy()
obs_overlap  = df_climate.loc[overlap_start:overlap_end].copy()

proj_overlap['dekad'] = dekad_of_year(proj_overlap.index)
obs_overlap['dekad']  = dekad_of_year(obs_overlap.index)

proj_by_dekad = proj_overlap.groupby('dekad')[['PCP_proj', 'RET_proj']].mean()
obs_by_dekad  = obs_overlap.groupby('dekad')[['PCP', 'RET']].mean()

# Ratio (WaPOR observed / MIROC6 simulated) per dekad-of-year -- this is the
# REAL bias: how far off MIROC6 is from reality, in the one window where we
# have both. Clipped to guard against near-zero MIROC6 dekads blowing the
# ratio up.
climate_bias = pd.DataFrame({
    'PCP_ratio': (obs_by_dekad['PCP'] / proj_by_dekad['PCP_proj']).clip(0.2, 5.0),
    'RET_ratio': (obs_by_dekad['RET'] / proj_by_dekad['RET_proj']).clip(0.2, 5.0),
})

print("="*70)
print("MIROC6 BIAS -- WaPOR observed vs MIROC6 simulated, 2018-2025, by dekad-of-year")
print("="*70)
print(f"   PCP ratio (WaPOR/MIROC6): mean = {climate_bias['PCP_ratio'].mean():.2f}, "
      f"range = [{climate_bias['PCP_ratio'].min():.2f}, {climate_bias['PCP_ratio'].max():.2f}]")
print(f"   RET ratio (WaPOR/MIROC6): mean = {climate_bias['RET_ratio'].mean():.2f}, "
      f"range = [{climate_bias['RET_ratio'].min():.2f}, {climate_bias['RET_ratio'].max():.2f}]")

comparison_table = pd.DataFrame({
    'PCP_MIROC6': proj_by_dekad['PCP_proj'].round(2),
    'PCP_WaPOR':  obs_by_dekad['PCP'].round(2),
    'PCP_diff_mm': (obs_by_dekad['PCP'] - proj_by_dekad['PCP_proj']).round(2),
    'PCP_pct':    ((climate_bias['PCP_ratio'] - 1) * 100).round(1),
    'RET_MIROC6': proj_by_dekad['RET_proj'].round(2),
    'RET_WaPOR':  obs_by_dekad['RET'].round(2),
    'RET_diff_mm': (obs_by_dekad['RET'] - proj_by_dekad['RET_proj']).round(2),
    'RET_pct':    ((climate_bias['RET_ratio'] - 1) * 100).round(1),
})
print("\nDekad-by-dekad comparison (2018-2025 overlap) - first 5 dekads:")
display(comparison_table.head())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(proj_by_dekad.index, proj_by_dekad['PCP_proj'], color='#595AE3',
            linestyle='--', label='MIROC6 (2018-2025)')
axes[0].plot(obs_by_dekad.index, obs_by_dekad['PCP'], color='#595AE3',
            label='WaPOR observed (2018-2025)')
axes[0].set_title('PCP -- mean by dekad-of-year', fontweight='bold')
axes[0].set_xlabel('Dekad of year (1-36)'); axes[0].set_ylabel('mm/dekad')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.25)

axes[1].plot(proj_by_dekad.index, proj_by_dekad['RET_proj'], color='#5B9453',
            linestyle='--', label='MIROC6 (2018-2025)')
axes[1].plot(obs_by_dekad.index, obs_by_dekad['RET'], color='#5B9453',
            label='WaPOR observed (2018-2025)')
axes[1].set_title('RET -- mean by dekad-of-year', fontweight='bold')
axes[1].set_xlabel('Dekad of year (1-36)'); axes[1].set_ylabel('mm/dekad')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

top_pcp = comparison_table['PCP_pct'].abs().idxmax()
top_ret = comparison_table['RET_pct'].abs().idxmax()

In [ ]:
# =============================================================================
# APPLY BIAS CORRECTION TO THE FULL PROJECTION
# =============================================================================

proj_dekad = dekad_of_year(proj.index)
pcp_factor = climate_bias['PCP_ratio'].reindex(proj_dekad).values
ret_factor = climate_bias['RET_ratio'].reindex(proj_dekad).values

proj_corrected = proj.copy()
proj_corrected['PCP_proj'] = (proj['PCP_proj'].values * pcp_factor).clip(min=0)
proj_corrected['RET_proj'] = (proj['RET_proj'].values * ret_factor).clip(min=0)

annual_raw       = proj['PCP_proj'].resample('YE').sum()
annual_corrected = proj_corrected['PCP_proj'].resample('YE').sum()

print("="*70)
print("BIAS-CORRECTED PROJECTION READY")
print("="*70)
print(f"   {len(proj_corrected):,} dekads, {proj_corrected.index.min().date()} -> "
      f"{proj_corrected.index.max().date()}")
print(f"   PCP annual mean -- raw: {annual_raw.mean():.0f} mm/yr, "
      f"bias-corrected: {annual_corrected.mean():.0f} mm/yr")

---

> ⏸️ **Stop here — let's take stock of what we just did.**
>
> MIROC6, like every GCM, has its own systematic bias — it doesn't perfectly
> reproduce Erbil's real climate. We don't have a way to check that bias
> against reality for the deep past (WaPOR only starts in 2018), but we *do*
> have eight years (2018-2025) where both WaPOR's real observations and
> MIROC6's simulated "future" overlap. That overlap is the only place we can
> actually measure how wrong MIROC6 is — so that's what the cell above did:
>
> 1. Compared MIROC6's simulated PCP/RET against what WaPOR really recorded,
>    dekad-of-year by dekad-of-year, over 2018-2025.
> 2. Turned that gap into a correction ratio for each of the 36 dekads.
> 3. Applied that ratio to the *entire* projection (2015-2099), producing
>    `proj_corrected` -- the version we'll actually feed into the AETI model.
>
> 💭 **Worth keeping in mind:** this correction is built from only 8 years of
> overlap, and it assumes MIROC6's bias stays roughly constant all the way
> out to 2099 (the standard assumption behind this kind of delta/ratio
> correction -- but still an assumption, not a guarantee). Check the table
> and chart above for the actual PCP/RET bias percentages before trusting
> the AETI results that follow.

In [ ]:
# =============================================================================
# PROJECTING AETI SARIMAX+PCP+RET
# =============================================================================

PROJ_HORIZON = proj.index

project_climate_results = {}

for i, pivot in enumerate(pivots):
    print(f"\r   Fitting pivot {pivot} ({i+1}/{n_pivots})...", end='', flush=True)
    train_y = train_aeti[pivot].dropna()
    train_x = train_climate.reindex(train_y.index).ffill().bfill()

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=ValueWarning)
            warnings.simplefilter("ignore", category=FutureWarning)

            model = SARIMAX(train_y, exog=train_x, order=order,
                            seasonal_order=seasonal_order,
                            enforce_stationarity=False, enforce_invertibility=False)
            fit = model.fit(disp=False, maxiter=300)
            project = fit.get_forecast(steps=len(PROJ_HORIZON), exog=proj_corrected)
            mean = project.predicted_mean.clip(lower=0); mean.index = PROJ_HORIZON
            ci   = project.conf_int();                   ci.index   = PROJ_HORIZON
        project_climate_results[pivot] = {'mean': mean, 'ci': ci}

    except Exception as e:
        print(f"\n   ⚠️ Pivot {pivot}: {e}")
        # Corrected: Update project_climate_results for failed pivots
        project_climate_results[pivot] = {'mean': None, 'ci': None}

### Annual AETI Projection with 10-year rolling mean

 ➜ Each pivot has ~3,100 individual 10-day (dekadal) data points spanning 86 years (2015-2099). Within each year, AETI drops to near-zero in the dry season (June-September, no crop = no water consumption) and peaks sharply in the wet season -- that's 36 rises and falls per year, repeated 86 times.

📈 Plotting that full resolution packs ~3,100 spikes side by side, which the eye can't distinguish from noise -- it reads as a solid red block ("comb effect"), and any real long-term trend gets buried underneath.

🧮 Summing each year's 36 dekads into a single annual total collapses that seasonal noise into one number per year (86 points instead of 3,100), leaving visible what actually matters: how much water the pivots consume per year, and how that's expected to change over the century.


In [ ]:
# =============================================================================
# ANNUAL AETI PROJECTION WITH 10-YEAR ROLLING MEAN
# =============================================================================

fig, axes = plt.subplots(n_rows_p, n_cols_p, figsize=(18, 3.5 * n_rows_p))
fig.suptitle('Future AETI Forecasts -- Annual Totals with 10-yr Rolling Mean',
             fontsize=16, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

for idx, pivot in enumerate(pivots):
    ax = axes_flat[idx]

    # Observed history, annual totals
    obs_annual = df_aeti[pivot].resample('YE').sum()
    ax.plot(obs_annual.index, obs_annual.values, color='steelblue',
            linewidth=1.2, marker='o', markersize=3, label='Observed (annual)')

    fc = project_climate_results[pivot]
    if fc['mean'] is not None:
        # Projected annual totals
        proj_annual = fc['mean'].resample('YE').sum()
        roll = proj_annual.rolling(10, min_periods=5).mean()

        # Annual CI band: sum dekadal bounds per year, consistent with how
        # the annual total itself is a sum of dekads
        ci = fc['ci']
        lower_col, upper_col = ci.columns[0], ci.columns[1]
        ci_annual_lower = ci[lower_col].clip(lower=0).resample('YE').sum()
        ci_annual_upper = ci[upper_col].resample('YE').sum()

        ax.fill_between(proj_annual.index, ci_annual_lower, ci_annual_upper,
                        color='#d62728', alpha=0.12, label='95% CI (annual)')
        ax.plot(proj_annual.index, proj_annual.values, color='#d62728',
                linewidth=0.6, alpha=0.4, label='Projected (annual)')
        ax.plot(roll.index, roll.values, color='#d62728', linewidth=2,
                label='10-yr rolling mean')
        total_future = proj_annual.mean()
    else:
        total_future = 0

    ax.axvline(x=df_aeti.index[-1], color='gray', linewidth=0.8, alpha=0.5)
    ax.set_title(f"P{pivot} (proj. mean {total_future:.0f} mm/yr)", fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.2)
    ax.set_ylim(bottom=0)

axes_flat[0].legend(fontsize=7, loc='upper left')
for idx in range(n_pivots, len(axes_flat)):
    axes_flat[idx].set_visible(False)
plt.tight_layout()
plt.show()

### Annual AETI Projection with 1-year rolling mean

Instead of collapsing to one point per year, this smooths the dekadal
series with a 36-dekad (1-year) moving average -- removes the within-year
seasonal spikes while keeping dekadal time resolution, so shorter-term
shifts stay visible (the 10-yr version above only shows the long-run trend).

In [ ]:
# =============================================================================
# ANNUAL AETI PROJECTION WITH 1-YEAR ROLLING MEAN (dekadal resolution)
# =============================================================================

fig, axes = plt.subplots(n_rows_p, n_cols_p, figsize=(18, 3.5 * n_rows_p))
fig.suptitle('Future AETI Forecasts -- 1-Year Rolling Mean (dekadal resolution)',
             fontsize=16, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

for idx, pivot in enumerate(pivots):
    ax = axes_flat[idx]

    obs_annual = df_aeti[pivot].resample('YE').sum()
    ax.plot(obs_annual.index, obs_annual.values, color='steelblue',
            linewidth=1.2, marker='o', markersize=3, label='Observed (annual)')

    fc = project_climate_results[pivot]
    if fc['mean'] is not None:
        roll_1yr = fc['mean'].rolling(36, min_periods=18).mean() * 36

        ci = fc['ci']
        lower_col, upper_col = ci.columns[0], ci.columns[1]
        ci_roll_lower = ci[lower_col].clip(lower=0).rolling(36, min_periods=18).mean() * 36
        ci_roll_upper = ci[upper_col].rolling(36, min_periods=18).mean() * 36

        ax.fill_between(roll_1yr.index, ci_roll_lower, ci_roll_upper,
                        color='#d62728', alpha=0.12, label='95% CI (1-yr rolling)')
        ax.plot(roll_1yr.index, roll_1yr.values, color='#d62728', linewidth=1.3,
                label='1-yr rolling mean')
        total_future = roll_1yr.mean()
    else:
        total_future = 0

    ax.axvline(x=df_aeti.index[-1], color='gray', linewidth=0.8, alpha=0.5)
    ax.set_title(f"P{pivot} (proj. mean {total_future:.0f} mm/yr)", fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.2)
    ax.set_ylim(bottom=0)

axes_flat[0].legend(fontsize=7, loc='upper left')
for idx in range(n_pivots, len(axes_flat)):
    axes_flat[idx].set_visible(False)
plt.tight_layout()
plt.show()

### 🌦️ Future Seasonality Shifts (PCP & RET)

This cell compares the seasonal cycles of Precipitation and Reference ET across three periods: **Historical Baseline**, **Near Future**, and **Far Future**.

**What it does:**
Data is grouped by **dekad-of-year** (10-day intervals) to reveal high-resolution shifts in seasonal timing.

**What to look for:**
Observe the curves to identify if wet or dry seasons are shifting (e.g., starting later) or changing in intensity over the century.

In [ ]:
# =============================================================================
# SEASONALITY ACROSS FUTURE PERIODS: PCP & RET, BY DEKAD-OF-YEAR
# =============================================================================

def dekad_climatology(df, start=None, end=None):
    """Mean PCP/RET by dekad-of-year, optionally restricted to a date range."""
    d = df.loc[start:end] if start else df
    d = d.copy()
    d['dekad'] = dekad_of_year(d.index)
    return d.groupby('dekad')[['PCP_proj', 'RET_proj']].mean()

hist_clim = dekad_climatology(proj_hist)
near_clim = dekad_climatology(proj_corrected, '2040-01-01', '2059-12-31')
far_clim  = dekad_climatology(proj_corrected, '2080-01-01', '2099-12-31')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(hist_clim.index, hist_clim['PCP_proj'], color='gray',
            linestyle=':', linewidth=1.5, label='Historical (1950-2014)')
axes[0].plot(near_clim.index, near_clim['PCP_proj'], color='#f4a261',
            linewidth=1.8, label='Near-future (2040-2060)')
axes[0].plot(far_clim.index,  far_clim['PCP_proj'],  color='#d62728',
            linewidth=1.8, label='Far-future (2080-2100)')
axes[0].set_title('PCP -- seasonal pattern by dekad-of-year', fontweight='bold')
axes[0].set_xlabel('Dekad of year (1-36)'); axes[0].set_ylabel('mm/dekad')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.25)

axes[1].plot(hist_clim.index, hist_clim['RET_proj'], color='gray',
            linestyle=':', linewidth=1.5, label='Historical (1950-2014)')
axes[1].plot(near_clim.index, near_clim['RET_proj'], color='#f4a261',
            linewidth=1.8, label='Near-future (2040-2060)')
axes[1].plot(far_clim.index,  far_clim['RET_proj'],  color='#d62728',
            linewidth=1.8, label='Far-future (2080-2100)')
axes[1].set_title('RET -- seasonal pattern by dekad-of-year', fontweight='bold')
axes[1].set_xlabel('Dekad of year (1-36)'); axes[1].set_ylabel('mm/dekad')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

### 🌍 Future AETI Seasonality Analysis

This cell compares our historical AETI baseline (2018-2025) against two future climate projections (2020-2060 and 2060-2100).

**What it does:**
It aggregates regional data to visualize future changes in water consumption:
* **Seasonal Curve:** 10-day averages (dekads) showing the detailed seasonal trend.
* **Monthly Totals:** Calendar month sums for a high-level volume comparison.

**What to look for:**
Observe the output charts to see if future projections (orange/red) indicate higher overall AETI or shifts in the peak seasons compared to the historical baseline (gray/blue).

In [ ]:
# =============================================================================
# AETI SEASONALITY ACROSS 40-YEAR FUTURE PERIODS -- REGIONAL MEAN
# =============================================================================
# Same idea as the PCP/RET seasonality comparison, applied to AETI: baseline
# (observed) vs two 40-year future blocks (2020-2060, 2060-2100), shown as
# dekad-of-year (left) and monthly (right) climatology, side by side.

NEAR_FUTURE = ('2020-01-01', '2059-12-31')   # first 40-year block
FAR_FUTURE  = ('2060-01-01', '2099-12-31')   # second 40-year block, through 2100

def aeti_dekad_climatology(series, start=None, end=None):
    """Mean AETI by dekad-of-year (mm/dekad), optionally restricted to a date range."""
    d = series.loc[start:end] if start else series
    return d.groupby(dekad_of_year(d.index)).mean()

def monthly_climatology(series, start=None, end=None):
    """Mean AETI per calendar month (mm/month), from a dekadal series."""
    d = series.loc[start:end] if start else series
    monthly = d.resample('ME').sum()
    return monthly.groupby(monthly.index.month).mean()

baseline_dekad, near_dekad, far_dekad = [], [], []
baseline_month, near_month, far_month = [], [], []

for pivot in pivots:
    baseline_dekad.append(aeti_dekad_climatology(df_aeti[pivot]))
    baseline_month.append(monthly_climatology(df_aeti[pivot]))
    fc = project_climate_results[pivot]
    if fc['mean'] is not None:
        near_dekad.append(aeti_dekad_climatology(fc['mean'], *NEAR_FUTURE))
        far_dekad.append(aeti_dekad_climatology(fc['mean'], *FAR_FUTURE))
        near_month.append(monthly_climatology(fc['mean'], *NEAR_FUTURE))
        far_month.append(monthly_climatology(fc['mean'], *FAR_FUTURE))

baseline_dekad_r = pd.concat(baseline_dekad, axis=1).mean(axis=1)
near_dekad_r      = pd.concat(near_dekad, axis=1).mean(axis=1)
far_dekad_r        = pd.concat(far_dekad, axis=1).mean(axis=1)

baseline_regional = pd.concat(baseline_month, axis=1).mean(axis=1)
near_regional      = pd.concat(near_month, axis=1).mean(axis=1)
far_regional        = pd.concat(far_month, axis=1).mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: dekad-of-year seasonal pattern
axes[0].plot(baseline_dekad_r.index, baseline_dekad_r.values, color='gray',
            linestyle=':', linewidth=1.5, label='Baseline (2018-2025 observed)')
axes[0].plot(near_dekad_r.index, near_dekad_r.values, color='#f4a261',
            linewidth=1.8, label='2020-2060')
axes[0].plot(far_dekad_r.index, far_dekad_r.values, color='#d62728',
            linewidth=1.8, label='2060-2100')
axes[0].set_title('AETI -- seasonal pattern by dekad-of-year', fontweight='bold')
axes[0].set_xlabel('Dekad of year (1-36)'); axes[0].set_ylabel('mm/dekad')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.25)

# Right: monthly climatology (bar chart)
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
x = np.arange(1, 12)
width = 0.26

axes[1].bar(x - width, baseline_regional.reindex(x).values, width,
       label='Baseline (2018-2025 observed)', color='steelblue')
axes[1].bar(x,          near_regional.reindex(x).values,     width,
       label='2020-2060', color='#f4a261')
axes[1].bar(x + width,  far_regional.reindex(x).values,      width,
       label='2060-2100', color='#d62728')
axes[1].set_xticks(x); axes[1].set_xticklabels(month_names)
axes[1].set_ylabel('AETI (mm/month)')
axes[1].set_title('AETI -- mean monthly totals', fontweight='bold')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.show()

---

<a id="12"></a>
## 12. Spatial View — Mapping Forecast Quality & Future Outlook

> *If a GeoJSON of the 12 pivots is available, we map forecast accuracy and future risk spatially.*

**⬇️ This section needs `SelectedCPs.geojson` (the pivot boundaries) uploaded to `/content/data/` before running the cells below.** Upload it in the cell right below -- without it, both maps are skipped and you'll just see an "ℹ️ GeoPandas or GeoJSON missing" message instead of the figures.

In [ ]:
# -----------------------------------------------------------------------------
# Upload the pivot boundaries GeoJSON used in this section
# -----------------------------------------------------------------------------
from google.colab import files
uploaded = files.upload()

for filename, content in uploaded.items():
    dest_path = f'/content/data/{filename}'
    with open(dest_path, 'wb') as f:
        f.write(content)
    print(f" Saved: {dest_path}")

In [ ]:
# =============================================================================
# DIFFERENCES: NEAR-FUTURE AND FAR-FUTURE vs BASELINE
# =============================================================================

rows = []
for pivot in pivots:
    baseline_annual = df_aeti[pivot].resample('YE').sum().mean()
    fc = project_climate_results[pivot]
    if fc['mean'] is None:
        continue
    near_annual = fc['mean'].loc[NEAR_FUTURE[0]:NEAR_FUTURE[1]].resample('YE').sum().mean()
    far_annual  = fc['mean'].loc[FAR_FUTURE[0]:FAR_FUTURE[1]].resample('YE').sum().mean()

    rows.append({
        'pivot': pivot,
        'baseline_mm_yr': round(baseline_annual, 0),
        'near_2020_2060_mm_yr': round(near_annual, 0),
        'near_diff_mm': round(near_annual - baseline_annual, 0),
        'near_diff_pct': round((near_annual / baseline_annual - 1) * 100, 1),
        'far_2060_2100_mm_yr': round(far_annual, 0),
        'far_diff_mm': round(far_annual - baseline_annual, 0),
        'far_diff_pct': round((far_annual / baseline_annual - 1) * 100, 1),
    })

diff_table = pd.DataFrame(rows).set_index('pivot')
diff_table.loc['REGIONAL MEAN'] = diff_table.mean()

print("="*70)
print("AETI: NEAR-FUTURE / FAR-FUTURE vs BASELINE (2018-2025 observed)")
print("="*70)
display(diff_table)

In [ ]:
# =============================================================================
# SECTION 11.5 SUMMARY DASHBOARD (STATIC)
# =============================================================================
# This cell pulls together variables built across the WHOLE section -- it
# needs everything from "READ CLIMATE PROJECTED DATA" through the
# differences table to have already run in this session. If you restarted
# the runtime, re-run Section 11.5 top to bottom before this cell.

required_vars = ['df_aeti', 'pivots', 'project_climate_results', 'baseline_regional',
                 'near_regional', 'far_regional', 'diff_table']
missing = [v for v in required_vars if v not in dir()]
if missing:
    print(f"⚠️  Missing variables: {missing}")
    print("   Run the earlier Section 11.5 cells first (bias correction, AETI")
    print("   projection, seasonality, and differences table) before this one.")

fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

fig.suptitle('Section 11.5 Summary -- Long-Term Climate Projection & AETI Impact',
             fontsize=17, fontweight='bold', y=0.98)

# --- Panel A (top-left): Regional annual AETI trend ---
ax1 = fig.add_subplot(gs[0, 0])
if 'project_climate_results' in dir():
    obs_regional_annual = pd.concat([df_aeti[p].resample('YE').sum() for p in pivots], axis=1).mean(axis=1)
    proj_regional_annual = pd.concat([project_climate_results[p]['mean'].resample('YE').sum()
                                      for p in pivots if project_climate_results[p]['mean'] is not None], axis=1).mean(axis=1)
    roll_regional = proj_regional_annual.rolling(10, min_periods=5).mean()

    ax1.plot(obs_regional_annual.index, obs_regional_annual.values, color='steelblue',
            marker='o', markersize=4, linewidth=1.2, label='Observed (regional mean)')
    ax1.plot(proj_regional_annual.index, proj_regional_annual.values, color='#d62728',
            linewidth=0.6, alpha=0.4, label='Projected (annual)')
    ax1.plot(roll_regional.index, roll_regional.values, color='#d62728', linewidth=2.2,
            label='10-yr rolling mean')
    ax1.axvline(df_aeti.index[-1], color='gray', linewidth=0.8, alpha=0.5)
    ax1.legend(fontsize=8)
else:
    ax1.text(0.5, 0.5, 'Run the AETI projection cells first', ha='center', va='center')
ax1.set_title('A. Regional Annual AETI Trend', fontweight='bold', fontsize=12)
ax1.set_ylabel('mm/yr'); ax1.grid(alpha=0.2)

# --- Panel B (top-right): Regional monthly climatology ---
ax2 = fig.add_subplot(gs[0, 1])
if 'baseline_regional' in dir():
    month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    x = np.arange(1, 13)
    width = 0.26
    ax2.bar(x - width, baseline_regional.reindex(x).values, width,
           label='Baseline (2018-2025)', color='steelblue')
    ax2.bar(x,          near_regional.reindex(x).values,     width,
           label='2020-2060', color='#f4a261')
    ax2.bar(x + width,  far_regional.reindex(x).values,      width,
           label='2060-2100', color='#d62728')
    ax2.set_xticks(x); ax2.set_xticklabels(month_names, fontsize=8)
    ax2.legend(fontsize=8)
else:
    ax2.text(0.5, 0.5, 'Run the seasonality cells first', ha='center', va='center')
ax2.set_title('B. Regional Mean Monthly AETI', fontweight='bold', fontsize=12)
ax2.set_ylabel('mm/month'); ax2.grid(alpha=0.2, axis='y')

# --- Panel C (bottom-left): Ranked bar chart by pivot ---
ax3 = fig.add_subplot(gs[1, 0])
if 'diff_table' in dir():
    plot_df = diff_table.drop('REGIONAL MEAN', errors='ignore').sort_values('far_diff_pct')
    y = np.arange(len(plot_df))
    ax3.barh(y - 0.2, plot_df['near_diff_pct'], height=0.35, label='2020-2060', color='#f4a261')
    ax3.barh(y + 0.2, plot_df['far_diff_pct'], height=0.35, label='2060-2100', color='#d62728')
    ax3.set_yticks(y); ax3.set_yticklabels([f'P{p}' for p in plot_df.index], fontsize=8)
    ax3.axvline(0, color='gray', linewidth=0.8)
    ax3.legend(fontsize=8)
else:
    ax3.text(0.5, 0.5, 'Run the differences table cell first', ha='center', va='center')
ax3.set_title('C. AETI Change by Pivot, Ranked', fontweight='bold', fontsize=12)
ax3.set_xlabel('% change vs baseline'); ax3.grid(alpha=0.2, axis='x')

# --- Panel D (bottom-right): Static spatial map, far-future ---
ax4 = fig.add_subplot(gs[1, 1])

# Import geopandas defensively, right here -- don't assume an earlier cell
# already did it successfully in this session.
try:
    import geopandas as gpd
    gpd_ok = True
except ImportError:
    gpd_ok = False

# Rebuild gdf_plot here if it isn't already in memory, so this panel doesn't
# silently depend on the earlier spatial-mapping cells having run.
if 'gdf_plot' not in dir() or gdf_plot is None or gdf_plot.empty:
    geojson_file = os.path.join(data_folder, 'SelectedCPs.geojson')
    if gpd_ok and os.path.exists(geojson_file) and 'diff_table' in dir():
        gdf = gpd.read_file(geojson_file)
        gdf['id_str'] = gdf['id'].astype(str)
        merge_rows = []
        for pivot in pivots:
            possible_ids = [str(int(pivot) + 1), str(pivot)]
            for tid in possible_ids:
                if tid in gdf['id_str'].values:
                    if pivot in diff_table.index:
                        merge_rows.append({
                            'pivot_geo_id': tid,
                            'orig_id': pivot,
                            'Near-future % change (2020-2060)': diff_table.loc[pivot, 'near_diff_pct'],
                            'Far-future % change (2060-2100)': diff_table.loc[pivot, 'far_diff_pct'],
                        })
                    break
        gdf_plot = gdf.merge(pd.DataFrame(merge_rows), left_on='id_str', right_on='pivot_geo_id', how='inner')
    else:
        gdf_plot = None

if gdf_plot is not None and not gdf_plot.empty:
    change_cols = ['Near-future % change (2020-2060)', 'Far-future % change (2060-2100)']
    vmax_map = np.nanpercentile(gdf_plot[change_cols].abs(), 90)

    if 'erbil_gdf' not in dir() and gpd_ok:
        erbil_file = os.path.join(data_folder, 'Erbil.geojson')
        erbil_gdf = gpd.read_file(erbil_file) if os.path.exists(erbil_file) else None
    if 'erbil_gdf' in dir() and erbil_gdf is not None:
        erbil_gdf.boundary.plot(ax=ax4, color='gray', linewidth=1)

    # Plot pivots as enlarged centroid markers instead of true-to-scale
    # polygons -- the real fields are tiny at this map scale to be visible.
    # Size is 60% larger than the previous default (200 -> 320).
    centroids = gdf_plot.geometry.centroid
    base_size = 200
    marker_size = base_size * 1.6

    sc = ax4.scatter(centroids.x, centroids.y,
                     c=gdf_plot['Far-future % change (2060-2100)'],
                     cmap='RdBu_r', vmin=-vmax_map, vmax=vmax_map,
                     s=marker_size, edgecolor='black', linewidth=0.8, zorder=3)

    cbar = plt.colorbar(sc, ax=ax4, shrink=0.7)
    cbar.set_label('% AETI change')

    # Label each pivot with just its ID -- no percentage text on the map,
    # magnitude comes across through color instead.
    for _, row in gdf_plot.iterrows():
        centroid = row['geometry'].centroid
        ax4.annotate(f"P{row['orig_id']}", (centroid.x, centroid.y),
                    fontsize=7, fontweight='bold', ha='center', va='center', zorder=4)

    minx, miny, maxx, maxy = gdf_plot.total_bounds
    pad_map = 0.05
    ax4.set_xlim(minx - pad_map, maxx + pad_map)
    ax4.set_ylim(miny - pad_map, maxy + pad_map)

    # Coordinate grid -- reads like a map, not a floating scatter
    ax4.axis('on')
    ax4.grid(True, linestyle=':', alpha=0.4)
    ax4.set_xlabel('Longitude', fontsize=9)
    ax4.set_ylabel('Latitude', fontsize=9)
    ax4.tick_params(labelsize=7)
else:
    reason = 'geopandas not available' if not gpd_ok else \
             "'diff_table' missing -- run that cell first" if 'diff_table' not in dir() else \
             'GeoJSON not found or merge came back empty'
    ax4.text(0.5, 0.5, reason, ha='center', va='center', fontsize=10, wrap=True)
    ax4.axis('off')
ax4.set_title('D. Spatial Distribution, Far-Future (2060-2100)', fontweight='bold', fontsize=12)

---
## Course Synthesis — The Three Zooms

| Topic | Scale | Key Insight |
|-------|-------|-------------|
| **1. Country** | 18 Governorates | Iraq's structural water deficit — rainfall declining, ET demand rising |
| **2. Governorate** | Erbil Cropland | Even in the north, crops meet only a fraction of atmospheric demand; drought recurs |
| **3. Field** | 12 Centre Pivots | Fields vary enormously; forecasting is feasible and enables drought risk + water productivity assessment |

> **The journey from climate to field tells a consistent story:** Iraq's water challenge is real at every scale. Time series analysis helps us understand the past (Topics 1-2), forecasting tools predict water consumption (Topic 3), and forward projections of AETI and NPP turn predictions into **actionable decision support** for irrigation and drought management.

---

### References

1. **FAO WaPOR**: https://wapor.apps.fao.org/
2. **SARIMA**: Box, G.E.P. & Jenkins, G.M. (1976). Time Series Analysis: Forecasting and Control.
3. **SARIMAX**: Hamilton, J.D. (1994). Time Series Analysis. Princeton University Press.

---

*End of Topic 3 — End of Course*

**Thank you for joining this journey from country to field!**